In [ ]:
from pathlib import Path
import subprocess,sys,json,traceback
src=Path('/kaggle/working/scale_source');src.mkdir(exist_ok=True)
for name,text in {'preprocess.py': '"""Turn each DICOM series into a canonical uint8 volume [n_slices, SIZE, SIZE].\n\nCanonical means: fixed physical pixel size, centre crop to a fixed field of view,\nstandard in-plane orientation, standard slice order and a per-series intensity window.\nThe same `load_series` is used for the training cache and for inference.\n\nUsage: python src/preprocess.py [train|test] [n_series_limit]\n"""\nimport os\nimport sys\nfrom multiprocessing import Pool\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport pydicom\n\nDATA = "/mnt/ssd/rsna_knee/data"\nWORK = "/mnt/ssd/rsna_knee/work"\n\nMM_PER_PX = 0.4\nSIZE = 384  # 153.6 mm field of view\nMAX_SLICES = 64\n\n# plane -> (normal axis, slice-order sign, (col axis, sign), (row axis, sign)) in LPS coordinates\n# Sagittal: slices right->left, image columns anterior->posterior, rows superior->inferior\n# Coronal:  slices anterior->posterior, columns patient-right->left, rows superior->inferior\n# Axial:    slices superior->inferior, columns patient-right->left, rows anterior->posterior\nCANON = {\n    0: (+1, (1, +1), (2, -1)),\n    1: (+1, (0, +1), (2, -1)),\n    2: (-1, (0, +1), (1, +1)),\n}\nPLANES = ["Sagittal", "Coronal", "Axial"]\n\n\ndef _read(path):\n    ds = pydicom.dcmread(path)\n    arr = ds.pixel_array.astype(np.float32)\n    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":\n        arr = arr.max() - arr\n    slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")\n    if slope is not None and inter is not None:\n        arr = arr * float(slope) + float(inter)\n    iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None\n    ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None\n    ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None\n    inst = float(ds.get("InstanceNumber") or 0)\n    frames = list(arr) if arr.ndim == 3 else [arr]\n    return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]\n\n\ndef _canon_slice(img, iop, ps, plane):\n    """Reorient one slice to the canonical in-plane orientation; returns (img, (row_mm, col_mm))."""\n    ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2  # no spacing: assume the image spans the crop\n    if iop is None:\n        return img, ps\n    r, c = np.array(iop[:3]), np.array(iop[3:])\n    _, (cax, csgn), (rax, rsgn) = CANON[plane]\n    if abs(r[cax]) < abs(r[rax]):  # rows and columns are swapped relative to canonical\n        img, r, c, ps = img.T, c, r, ps[::-1]\n    if r[cax] * csgn < 0:\n        img = img[:, ::-1]\n    if c[rax] * rsgn < 0:\n        img = img[::-1]\n    return img, ps\n\n\ndef _resample(img, ps):\n    """Centre crop to SIZE*MM_PER_PX mm, resample to MM_PER_PX, zero-pad to SIZE x SIZE."""\n    out = np.zeros((SIZE, SIZE), np.float32)\n    h, w = img.shape\n    ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))\n    cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))\n    y0, x0 = (h - ch) // 2, (w - cw) // 2\n    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])\n    nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))\n    nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))\n    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR\n    res = cv2.resize(crop, (nw, nh), interpolation=interp)\n    oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2\n    out[oy:oy + nh, ox:ox + nw] = res\n    return out\n\n\ndef load_series(sdir, plane_name=None):\n    """Returns (uint8 volume [n, SIZE, SIZE], info dict). Unreadable slices are skipped."""\n    slices, bad = [], 0\n    for fn in os.listdir(sdir):\n        try:\n            slices.extend(_read(os.path.join(sdir, fn)))\n        except Exception:\n            bad += 1\n    info = {"n_raw": len(slices), "n_bad": bad}\n    if not slices:\n        return np.zeros((1, SIZE, SIZE), np.uint8), info\n\n    iop = next((s[1] for s in slices if s[1] is not None), None)\n    if iop is not None:\n        normal = np.cross(iop[:3], iop[3:])\n        plane = int(np.abs(normal).argmax())\n    else:\n        normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0\n    info["plane"] = PLANES[plane]\n\n    if normal is not None and all(s[2] is not None for s in slices):\n        sgn = CANON[plane][0] * np.sign(normal[plane])\n        slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)\n        pos = [float(np.dot(s[2], normal)) for s in slices]\n        info["slice_gap"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 0.0\n    else:\n        slices.sort(key=lambda s: s[4])\n    if len(slices) > MAX_SLICES:\n        keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)\n        slices = [slices[i] for i in keep]\n\n    vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])\n    lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])\n    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)\n    info["n"] = len(vol)\n    return (vol * 255).round().astype(np.uint8), info\n\n\ndef _job(args):\n    split, study, series, plane, out_dir = args\n    out = f"{out_dir}/{series}.npy"\n    info = {"study": study, "series": series}\n    if os.path.exists(out):\n        return None\n    try:\n        vol, extra = load_series(f"{DATA}/{split}_series/{study}/{series}", plane)\n        np.save(out + ".tmp.npy", vol)\n        os.replace(out + ".tmp.npy", out)\n        info.update(extra)\n    except Exception as e:\n        info["error"] = f"{type(e).__name__}: {e}"[:200]\n    return info\n\n\ndef main():\n    split = sys.argv[1] if len(sys.argv) > 1 else "train"\n    limit = int(sys.argv[2]) if len(sys.argv) > 2 else None\n    out_dir = f"{WORK}/cache{SIZE}_{split}"\n    os.makedirs(out_dir, exist_ok=True)\n    se = pd.read_csv(f"{DATA}/{split}_series.csv")\n    if limit:\n        se = se.sample(limit, random_state=0)\n    tasks = [(split, a, b, p, out_dir) for a, b, p in zip(se.StudyInstanceUID, se.SeriesInstanceUID, se.Anatomical_Plane)]\n    rows = []\n    with Pool(18) as p:\n        for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=4)):\n            if r:\n                rows.append(r)\n            if i % 1000 == 0:\n                print(f"{i}/{len(tasks)}", flush=True)\n    df = pd.DataFrame(rows)\n    meta = f"{WORK}/cache{SIZE}_{split}_meta.csv"\n    if os.path.exists(meta) and len(df):\n        df = pd.concat([pd.read_csv(meta), df]).drop_duplicates("series", keep="last")\n    if len(df):\n        df.to_csv(meta, index=False)\n        print("done", len(df), "errors:", int(df["error"].notna().sum()) if "error" in df else 0)\n\n\nif __name__ == "__main__":\n    main()', 'knee.py': '"""Shared dataset and model code for training and inference."""\nimport math\n\nimport numpy as np\nimport timm\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n          "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nPLANES = ["Sagittal", "Coronal", "Axial"]\n# slot = plane index + 3 * (1 - fat_suppressed): fat-suppressed planes first\nN_SLOTS = 6\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\ndef slot_of(plane, fat_sup):\n    return PLANES.index(plane) + 3 * (1 - int(fat_sup))\n\n\ndef build_study_table(series_df, n_slices):\n    """series_df: StudyInstanceUID, SeriesInstanceUID, Fat_Suppression, Anatomical_Plane.\n    n_slices: dict series -> cached slice count. Returns {study: [[series ids of slot 0], ...]}\n    with each slot\'s candidates sorted by slice count (largest first)."""\n    table = {}\n    for st, sr, fs, pl in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID,\n                              series_df.Fat_Suppression, series_df.Anatomical_Plane):\n        if n_slices.get(sr, 0) < 3:\n            continue\n        table.setdefault(st, [[] for _ in range(N_SLOTS)])[slot_of(pl, fs)].append(sr)\n    for slots in table.values():\n        for cands in slots:\n            cands.sort(key=lambda s: -n_slices[s])\n    return table\n\n\ndef window_centres(n, k, train, lo=0.04, hi=0.96):\n    """k slice indices (centres of 3-slice windows) spread over the series."""\n    a, b = max(1.0, lo * (n - 1)), min(n - 2.0, hi * (n - 1))\n    if b < a:\n        a = b = (n - 1) / 2\n    if train:\n        edges = np.linspace(a, b, k + 1)\n        c = edges[:-1] + np.random.rand(k) * (edges[1:] - edges[:-1])\n    else:\n        c = np.linspace(a, b, k)\n    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))\n\n\nclass StudyDataset(torch.utils.data.Dataset):\n    """One item = one study: uint8 windows [N_SLOTS, k, 3, H, W], slot mask, slice positions, target."""\n\n    def __init__(self, studies, table, cache_dir, targets=None, k=8, train=False, loader=None):\n        self.studies, self.table, self.cache_dir = list(studies), table, cache_dir\n        self.targets, self.k, self.train = targets, k, train\n        self.loader = loader or (lambda s: np.load(f"{self.cache_dir}/{s}.npy", mmap_mode="r"))\n\n    def __len__(self):\n        return len(self.studies)\n\n    def __getitem__(self, i):\n        st = self.studies[i]\n        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])\n        x, mask, pos = None, np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)\n        for s, cands in enumerate(slots):\n            if not cands:\n                continue\n            sr = cands[np.random.randint(len(cands))] if self.train else cands[0]\n            vol = self.loader(sr)\n            n = len(vol)\n            c = window_centres(n, self.k, self.train)\n            win = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])  # [k, 3, H, W]\n            if x is None:\n                x = np.zeros((N_SLOTS, self.k) + win.shape[1:], np.uint8)\n            x[s], mask[s], pos[s] = win, True, c / max(n - 1, 1)\n        if x is None:\n            x = np.zeros((N_SLOTS, self.k, 3, 384, 384), np.uint8)\n        y = self.targets[i] if self.targets is not None else np.zeros(len(LABELS), np.float32)\n        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(np.asarray(y, np.float32))\n\n\ndef make_views(x, res, train, group, fov=0.92):\n    """x: uint8 [N, 3, H, W] with N = n_series * group. Crops a centred `fov` fraction of the cached\n    field of view and resizes to res; in training adds rotation/scale/shift and gain/gamma jitter\n    that is shared within a series."""\n    n = x.shape[0]\n    ns = n // group\n    dev = x.device\n    x = x.float().div_(255.0)\n    if train:\n        ang = (torch.rand(ns, device=dev) - 0.5) * 2 * math.radians(12)\n        sc = fov * (1 + (torch.rand(ns, device=dev) - 0.5) * 0.24)\n        tx = (torch.rand(ns, device=dev) - 0.5) * 0.12\n        ty = (torch.rand(ns, device=dev) - 0.5) * 0.12\n        gamma = torch.exp((torch.rand(ns, device=dev) - 0.5) * 0.5)\n        gain = 1 + (torch.rand(ns, device=dev) - 0.5) * 0.2\n    else:\n        ang = torch.zeros(ns, device=dev)\n        sc = torch.full((ns,), fov, device=dev)\n        tx = ty = torch.zeros(ns, device=dev)\n        gamma = gain = None\n    cos, sin = torch.cos(ang) * sc, torch.sin(ang) * sc\n    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)\n    theta = theta.repeat_interleave(group, 0)\n    grid = F.affine_grid(theta, (n, 3, res, res), align_corners=False)\n    if x.shape[-1] > 1.5 * res:  # avoid aliasing when shrinking a lot\n        x = F.avg_pool2d(x, 2) if x.shape[-1] >= 2 * res else F.interpolate(x, scale_factor=0.75, mode="bilinear", antialias=True)\n    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n    if train:\n        g = gamma.repeat_interleave(group).view(n, 1, 1, 1)\n        x = x.clamp_min(1e-4).pow(g) * gain.repeat_interleave(group).view(n, 1, 1, 1)\n    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)\n    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)\n    return (x - mean) / std\n\n\nclass KneeNet(nn.Module):\n    """2D backbone on 3-slice windows -> tokens tagged with series slot and slice position ->\n    small transformer over all windows of the study -> one attention pooling per finding."""\n\n    def __init__(self, backbone="convnext_tiny", pretrained=True, dim=384, depth=2, drop=0.2, drop_path=0.1,\n                 img_size=None):\n        super().__init__()\n        kw = {"drop_path_rate": drop_path} if drop_path else {}\n        if img_size and backbone.startswith(("vit_", "eva", "beit")):  # fixed-grid transformers need the input size\n            kw["img_size"] = img_size\n        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kw)\n        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim), nn.LayerNorm(dim))\n        self.slot_emb = nn.Embedding(N_SLOTS, dim)\n        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))\n        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)\n        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None\n        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, len(LABELS)))\n        self.cls_w = nn.Parameter(torch.randn(len(LABELS), dim) * 0.02)\n        self.cls_b = nn.Parameter(torch.zeros(len(LABELS)))\n\n    def head(self, feat, mask, pos):\n        """feat [B, S, K, D], mask [B, S] bool, pos [B, S, K] in 0..1 -> logits [B, 12]."""\n        b, s, k, _ = feat.shape\n        slot = torch.arange(s, device=feat.device).view(1, s, 1).expand(b, s, k)\n        h = self.proj(feat) + self.slot_emb(slot) + self.pos_emb(pos.unsqueeze(-1).to(feat.dtype))\n        h = h.view(b, s * k, -1)\n        pad = ~mask.unsqueeze(-1).expand(b, s, k).reshape(b, s * k)\n        pad = pad & ~pad.all(1, keepdim=True)  # a study with no series at all attends to everything\n        if self.tf is not None:\n            h = self.tf(h, src_key_padding_mask=pad)\n        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)  # [B, T, C]\n        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)\n        return (pooled * self.cls_w).sum(-1) + self.cls_b\n\n    def forward(self, x, mask, pos, res=256, train=False):\n        """x uint8 [B, S, K, 3, H, W]."""\n        b, s, k = x.shape[:3]\n        keep = mask.view(-1)  # run the backbone only on series that exist\n        xv = x.view(b * s, k, *x.shape[3:])[keep].flatten(0, 1)\n        f = self.enc(make_views(xv, res, train, k).contiguous(memory_format=torch.channels_last))\n        feat = f.new_zeros(b * s, k, f.shape[-1])\n        feat[keep] = f.view(-1, k, f.shape[-1])\n        return self.head(feat.view(b, s, k, -1), mask, pos)', 'infer.py': '"""Predict the 12 findings for every study straight from DICOM folders.\n\nUsed by the Kaggle submission notebook and locally to check that predictions from raw DICOMs\nmatch those from the training cache.\n\nUsage: python src/infer.py <data_dir> <split> <out_csv> <ckpt> [<ckpt> ...]\n  data_dir contains <split>.csv, <split>_series.csv and <split>_series/\n"""\nimport os\nimport sys\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom knee import LABELS, N_SLOTS, KneeNet, build_study_table, window_centres\nfrom preprocess import MAX_SLICES, SIZE, load_series\n\n\nclass DicomStudyDataset(torch.utils.data.Dataset):\n    """Loads every selected series of a study once; windows are cut per model setting later."""\n\n    def __init__(self, studies, table, root, k):\n        self.studies, self.table, self.root, self.k = list(studies), table, root, k\n\n    def __len__(self):\n        return len(self.studies)\n\n    def __getitem__(self, i):\n        st = self.studies[i]\n        slots = self.table.get(st, [[] for _ in range(N_SLOTS)])\n        x = np.zeros((N_SLOTS, self.k, 3, SIZE, SIZE), np.uint8)\n        mask, pos = np.zeros(N_SLOTS, bool), np.zeros((N_SLOTS, self.k), np.float32)\n        for s, cands in enumerate(slots):\n            if not cands:\n                continue\n            vol, info = load_series(f"{self.root}/{st}/{cands[0]}")\n            if info.get("n_bad", 0):\n                raise RuntimeError(f"DICOM decoding failed: {st}/{cands[0]}: {info}")\n            n = len(vol)\n            if n < 3:\n                raise RuntimeError(f"Selected series has fewer than 3 valid slices: {st}/{cands[0]}")\n            c = window_centres(n, self.k, False)\n            x[s] = np.stack([vol[np.clip([j - 1, j, j + 1], 0, n - 1)] for j in c])\n            mask[s], pos[s] = True, c / max(n - 1, 1)\n        if not mask.any():\n            raise RuntimeError(f"Study has no usable series: {st}")\n        return torch.from_numpy(x), torch.from_numpy(mask), torch.from_numpy(pos)\n\n\ndef load_models(ckpts, device="cuda"):\n    models = []\n    for path in ckpts:\n        ck = torch.load(path, map_location="cpu", weights_only=True)\n        # the pretrained tag after the dot is irrelevant here and unknown to older timm versions\n        m = KneeNet(ck["args"]["backbone"].split(".")[0], pretrained=False, img_size=ck["args"]["res"])\n        m.load_state_dict(ck["model"], strict=True)\n        models.append((m.to(device).to(memory_format=torch.channels_last).eval(), ck["args"]["res"]))\n    return models\n\n\n@torch.no_grad()\ndef run(data_dir, split, ckpts, k=12, studies=None, workers=4, bs=4):\n    se = pd.read_csv(f"{data_dir}/{split}_series.csv")\n    root = f"{data_dir}/{split}_series"\n    if studies is None:\n        studies = pd.read_csv(f"{data_dir}/{split}.csv").StudyInstanceUID.tolist()\n    se = se[se.StudyInstanceUID.isin(set(studies))]\n    n_files = {}\n    for st, sr in zip(se.StudyInstanceUID, se.SeriesInstanceUID):\n        d = f"{root}/{st}/{sr}"\n        n_files[sr] = min(len(os.listdir(d)), MAX_SLICES) if os.path.isdir(d) else 0\n    table = build_study_table(se, n_files)\n    dl = torch.utils.data.DataLoader(DicomStudyDataset(studies, table, root, k), batch_size=bs,\n                                     num_workers=workers, pin_memory=True)\n    models = load_models(ckpts)\n    out = []\n    for x, mask, pos in dl:\n        x, mask, pos = x.cuda(non_blocking=True), mask.cuda(), pos.cuda()\n        with torch.autocast("cuda", dtype=torch.float16):\n            p = torch.stack([m(x, mask, pos, res=res).float().sigmoid() for m, res in models]).mean(0)\n        out.append(p.cpu())\n    return pd.DataFrame(torch.cat(out).numpy(), index=pd.Index(studies, name="StudyInstanceUID"), columns=LABELS)\n\n\nif __name__ == "__main__":\n    data_dir, split, out_csv, ckpts = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4:]\n    run(data_dir, split, ckpts).to_csv(out_csv)\n    print("wrote", out_csv)', 'worker.py': '"""Isolated public reader execution; imported modules belong to this process only."""\nimport glob\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\nstarted=time.monotonic()\nsrc=Path(__file__).resolve().parent\nroots=[p.parent for p in Path(\'/kaggle/input\').glob(\'datasets/*/*/cnxt_v0_fold0.pt\')]\nroots += [p.parent for p in Path(\'/kaggle/input\').glob(\'*/cnxt_v0_fold0.pt\')]\nassert len(set(roots))==1, \'Checkpoint root must be unique\'\nasset=roots[0]\nckpts=[asset/f\'cnxt_v0_fold{i}.pt\' for i in range(3)]\nassert all(p.is_file() and p.stat().st_size==123035271 for p in ckpts), \'All three exact weights required\'\nenv=Path(\'/kaggle/working/cnx3_env\')\nsubprocess.run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-index\',\'--no-deps\',\'--target\',str(env),\'--find-links\',str(asset),\'pylibjpeg\',\'pylibjpeg-libjpeg\',\'pylibjpeg-openjpeg\'],check=True)\ntimm_wheels=sorted(Path(\'/kaggle/input\').glob(\'datasets/*/*/timm-*.whl\'))+sorted(Path(\'/kaggle/input\').glob(\'*/timm-*.whl\'))\nassert timm_wheels, \'Offline timm wheel required\'\nsubprocess.run([sys.executable,\'-m\',\'pip\',\'install\',\'-q\',\'--no-index\',\'--no-deps\',\'--target\',str(env),str(timm_wheels[0])],check=True)\nsys.path.insert(0,str(env))\nsys.path.insert(0,str(src))\nimport torch\nimport numpy as np\nimport pandas as pd\nimport infer\nfrom knee import LABELS, build_study_table\nfrom preprocess import MAX_SLICES\ntorch.set_num_threads(4)\ntorch.cuda.set_device(0)\nallocator_warmup=torch.empty(1,device=\'cuda:0\')\ndel allocator_warmup\ntorch.cuda.reset_peak_memory_stats(0)\ncomp=Path(\'/kaggle/input/competitions/rsna-knee-abnormality-detection\')\nif not comp.is_dir(): comp=Path(\'/kaggle/input/rsna-knee-abnormality-detection\')\ntest=pd.read_csv(comp/\'train.csv\',dtype={\'StudyInstanceUID\':str})\nids=[\'1.2.826.0.1.3680043.8.498.96425049974772570897047688166336162674\', \'1.2.826.0.1.3680043.8.498.38005916106577944091708997377832932938\', \'1.2.826.0.1.3680043.8.498.38554121831397903041461691318968151267\', \'1.2.826.0.1.3680043.8.498.12366238437963147962850341556508931973\', \'1.2.826.0.1.3680043.8.498.55344353927048912237233441917779875587\', \'1.2.826.0.1.3680043.8.498.11935893582662912138522962760478513770\', \'1.2.826.0.1.3680043.8.498.38935694477580522005505074220300062204\', \'1.2.826.0.1.3680043.8.498.13319886927127591971322856062062654146\', \'1.2.826.0.1.3680043.8.498.11629659616217352895144469980856259097\', \'1.2.826.0.1.3680043.8.498.12510856755988712246013743789183840748\', \'1.2.826.0.1.3680043.8.498.67284681623851829916403311739987186159\', \'1.2.826.0.1.3680043.8.498.11937911933281010146897685679862930140\', \'1.2.826.0.1.3680043.8.498.11049882927267755769995376553330493298\', \'1.2.826.0.1.3680043.8.498.81495303185504163367736663647489556517\', \'1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930\', \'1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542\', \'1.2.826.0.1.3680043.8.498.37713008080610827760634302982073393454\', \'1.2.826.0.1.3680043.8.498.80514754572353458107286859773300865138\', \'1.2.826.0.1.3680043.8.498.80556515221794602430701573588498144286\', \'1.2.826.0.1.3680043.8.498.12305733049746414359748178404911549573\', \'1.2.826.0.1.3680043.8.498.13879163301346984697246311226647565712\', \'1.2.826.0.1.3680043.8.498.59527896591755229337596505685198957637\', \'1.2.826.0.1.3680043.8.498.38173249623713271734596884672770674290\', \'1.2.826.0.1.3680043.8.498.63683477631564232018930819747238111809\', \'1.2.826.0.1.3680043.8.498.52937412433120578046842194545406797236\', \'1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941\', \'1.2.826.0.1.3680043.8.498.10419993051271996183722192755371692029\', \'1.2.826.0.1.3680043.8.498.12946163460972553911280889205685745877\', \'1.2.826.0.1.3680043.8.498.38457315909459574570206119213798163082\', \'1.2.826.0.1.3680043.8.498.10418888172081069709796646795446885715\', \'1.2.826.0.1.3680043.8.498.98112373403331748422888362310572528002\', \'1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839\', \'1.2.826.0.1.3680043.8.498.16158812105029662172517682642229718094\', \'1.2.826.0.1.3680043.8.498.11904156585987383639134967886361403369\', \'1.2.826.0.1.3680043.8.498.10427893579790102216163565692202919399\', \'1.2.826.0.1.3680043.8.498.98824066826946884510314135448574304233\', \'1.2.826.0.1.3680043.8.498.77244685031928454051409193776790348219\', \'1.2.826.0.1.3680043.8.498.13155572156594271049520969168607160312\', \'1.2.826.0.1.3680043.8.498.16589468854653713035210251187012168350\', \'1.2.826.0.1.3680043.8.498.31421989235441217674752676866539112529\', \'1.2.826.0.1.3680043.8.498.62030472630193758412620945408364583008\', \'1.2.826.0.1.3680043.8.498.47094981922544354899183959858078311972\', \'1.2.826.0.1.3680043.8.498.11920026019322352546372153111621676066\', \'1.2.826.0.1.3680043.8.498.13129258242739880102472095242703169773\', \'1.2.826.0.1.3680043.8.498.96859317329616917022799512740280754327\', \'1.2.826.0.1.3680043.8.498.98143909986671737130865120569673002676\', \'1.2.826.0.1.3680043.8.498.68769474672202195228430674317444754835\', \'1.2.826.0.1.3680043.8.498.29943089198208468441240036404506927883\', \'1.2.826.0.1.3680043.8.498.95882731821729917461275694847968503951\', \'1.2.826.0.1.3680043.8.498.23911658810234305013984976739924353704\', \'1.2.826.0.1.3680043.8.498.21556073772857867149576001337847113661\', \'1.2.826.0.1.3680043.8.498.33273874534046658504935877940537027702\', \'1.2.826.0.1.3680043.8.498.65977711270942843864266358352913175672\', \'1.2.826.0.1.3680043.8.498.79245771997885598399203314109623928718\', \'1.2.826.0.1.3680043.8.498.11920531053482540304184657218030242597\', \'1.2.826.0.1.3680043.8.498.25766673238950703352855821505938941878\', \'1.2.826.0.1.3680043.8.498.99178010550062268668346041362629896201\', \'1.2.826.0.1.3680043.8.498.32109377199577933060124357239881959602\', \'1.2.826.0.1.3680043.8.498.12050415099056648944934039580585859179\', \'1.2.826.0.1.3680043.8.498.86566584776908143877603164704838586097\', \'1.2.826.0.1.3680043.8.498.10190506315646895064803936408630113507\', \'1.2.826.0.1.3680043.8.498.38870467957974446108337293912930572060\', \'1.2.826.0.1.3680043.8.498.12640439197386154384526090653248054582\', \'1.2.826.0.1.3680043.8.498.95984406324600336713471618620285270466\', \'1.2.826.0.1.3680043.8.498.52046170319252016846878355722404423329\', \'1.2.826.0.1.3680043.8.498.91346221361264140052760282748336176394\', \'1.2.826.0.1.3680043.8.498.65013994414976103275014874284064145461\', \'1.2.826.0.1.3680043.8.498.12215902620611618183321984433114460986\', \'1.2.826.0.1.3680043.8.498.42057349781448469704191384429694247725\', \'1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833\', \'1.2.826.0.1.3680043.8.498.13229563888378977350299319159748823477\', \'1.2.826.0.1.3680043.8.498.94335780236370744781443631231792792083\', \'1.2.826.0.1.3680043.8.498.52844966504845792334740560021996858939\', \'1.2.826.0.1.3680043.8.498.60227430483444032531479844365716485643\', \'1.2.826.0.1.3680043.8.498.26023120405608635089544039694812003604\', \'1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548\', \'1.2.826.0.1.3680043.8.498.36235892729383338522021416304853301198\', \'1.2.826.0.1.3680043.8.498.34109091435857467600499949506940613037\', \'1.2.826.0.1.3680043.8.498.51676679394093588751371807453793655464\', \'1.2.826.0.1.3680043.8.498.13096288495834275184930201615129174987\', \'1.2.826.0.1.3680043.8.498.54374735423703474541190464716308417985\', \'1.2.826.0.1.3680043.8.498.86651345701812229172985086531423369421\', \'1.2.826.0.1.3680043.8.498.86263264208309369957750915520115390972\', \'1.2.826.0.1.3680043.8.498.42768603719459834345001093623865167866\', \'1.2.826.0.1.3680043.8.498.83237706274929081175032033288272528919\', \'1.2.826.0.1.3680043.8.498.80313580693550912024103221258669390487\', \'1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522\', \'1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525\', \'1.2.826.0.1.3680043.8.498.39845300426582562781355154625767485381\', \'1.2.826.0.1.3680043.8.498.36431753977367847587395456605556703040\', \'1.2.826.0.1.3680043.8.498.11770690655120964721059321709525979950\', \'1.2.826.0.1.3680043.8.498.37921127120678939078868474340027123335\', \'1.2.826.0.1.3680043.8.498.38974023308452365783768081643215200948\', \'1.2.826.0.1.3680043.8.498.23523176367321534630723586470190430254\', \'1.2.826.0.1.3680043.8.498.28023795363827851445172311810918021563\', \'1.2.826.0.1.3680043.8.498.12785620442256422697216192462703273802\', \'1.2.826.0.1.3680043.8.498.71660070729774772658965427776586988585\', \'1.2.826.0.1.3680043.8.498.10848350333764008140459854470293737650\', \'1.2.826.0.1.3680043.8.498.53393741562201846141435914434030220216\', \'1.2.826.0.1.3680043.8.498.82579118531566493365068788437906656162\', \'1.2.826.0.1.3680043.8.498.79829033430619597320769579608450393290\', \'1.2.826.0.1.3680043.8.498.12010150865087936209475730225377000084\', \'1.2.826.0.1.3680043.8.498.10391178814963564201551003358372148419\', \'1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662\', \'1.2.826.0.1.3680043.8.498.64034149345689391514924451724232196567\', \'1.2.826.0.1.3680043.8.498.57084445429567494093960261980941917499\', \'1.2.826.0.1.3680043.8.498.75418637314688036152637525301534342254\', \'1.2.826.0.1.3680043.8.498.66210455934011506415244876770725282769\', \'1.2.826.0.1.3680043.8.498.52558822670236188507569589448789513993\', \'1.2.826.0.1.3680043.8.498.79505118958475739396320647697583979079\', \'1.2.826.0.1.3680043.8.498.40718200801895968234499223301900799376\', \'1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461\', \'1.2.826.0.1.3680043.8.498.12917504242141830416624247604384969664\', \'1.2.826.0.1.3680043.8.498.49500752658775616590911569693817273205\', \'1.2.826.0.1.3680043.8.498.65809115193576546782858254430325338674\', \'1.2.826.0.1.3680043.8.498.12266133889557145601371064833389800065\', \'1.2.826.0.1.3680043.8.498.17967978373588651520812399064902037467\', \'1.2.826.0.1.3680043.8.498.71836684300308056297405632410209248276\', \'1.2.826.0.1.3680043.8.498.59048646289883992375610835845636263714\', \'1.2.826.0.1.3680043.8.498.49575149159425506211571562732929634662\', \'1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669\', \'1.2.826.0.1.3680043.8.498.79741629769872115526056483024289600492\', \'1.2.826.0.1.3680043.8.498.12499208731138180053966892599139021038\', \'1.2.826.0.1.3680043.8.498.89302191571937190529967168726521003428\', \'1.2.826.0.1.3680043.8.498.68862684196934468400953020419415008157\', \'1.2.826.0.1.3680043.8.498.11040296449332764126530525726037541310\', \'1.2.826.0.1.3680043.8.498.11079556243110299049578981210837702995\', \'1.2.826.0.1.3680043.8.498.95091624999382167943525022414631622532\', \'1.2.826.0.1.3680043.8.498.94594009911808653224418094594012527944\', \'1.2.826.0.1.3680043.8.498.36193046155628980489031133466840036976\', \'1.2.826.0.1.3680043.8.498.66083855405250624607521810369199218887\', \'1.2.826.0.1.3680043.8.498.63894147741855851067411708837856142566\', \'1.2.826.0.1.3680043.8.498.85890890545815190189345870912381925290\', \'1.2.826.0.1.3680043.8.498.93728117237268000947508816107096388086\', \'1.2.826.0.1.3680043.8.498.51426696939581135481763379246368970360\', \'1.2.826.0.1.3680043.8.498.22137222939487357840166329770642627189\', \'1.2.826.0.1.3680043.8.498.23686732555936573712275826587678502899\', \'1.2.826.0.1.3680043.8.498.10540882690103231159064894952851205573\', \'1.2.826.0.1.3680043.8.498.63377901538320536934403969965624202519\', \'1.2.826.0.1.3680043.8.498.99578894341571394220300473118855007095\', \'1.2.826.0.1.3680043.8.498.39064406065534097226924475518005522427\', \'1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080\', \'1.2.826.0.1.3680043.8.498.93653421810951484288813767943453706233\', \'1.2.826.0.1.3680043.8.498.61956863718716027369068406125965902569\', \'1.2.826.0.1.3680043.8.498.13042847643086819857036458569632034999\', \'1.2.826.0.1.3680043.8.498.25792216289349936450377008453709520260\', \'1.2.826.0.1.3680043.8.498.11542298537902846156898976374522470463\', \'1.2.826.0.1.3680043.8.498.66364462115470823900176770413527217503\', \'1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428\', \'1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831\', \'1.2.826.0.1.3680043.8.498.94625454484290996254750481989036812516\', \'1.2.826.0.1.3680043.8.498.34895742092137516664772577251243433579\', \'1.2.826.0.1.3680043.8.498.68328948893840135511720724977892905019\', \'1.2.826.0.1.3680043.8.498.31949224274318638796196151105617844024\', \'1.2.826.0.1.3680043.8.498.84419297518873367123595080276985526259\', \'1.2.826.0.1.3680043.8.498.73616237718955147476241731466078978343\', \'1.2.826.0.1.3680043.8.498.30282025087399390929963075109487390046\', \'1.2.826.0.1.3680043.8.498.84630493975910180537831353884518764364\', \'1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602\', \'1.2.826.0.1.3680043.8.498.38795933423959135237935740989825843893\', \'1.2.826.0.1.3680043.8.498.12968535455086298222684461157825070395\', \'1.2.826.0.1.3680043.8.498.87262205711949358943515326577886508585\', \'1.2.826.0.1.3680043.8.498.21868734450232158209414868901539475548\', \'1.2.826.0.1.3680043.8.498.11641866550826314411250696478180861612\', \'1.2.826.0.1.3680043.8.498.88457076333783710706782784856816112502\', \'1.2.826.0.1.3680043.8.498.74177736542522869266952610634091628727\', \'1.2.826.0.1.3680043.8.498.73477249559479870799427355988138990349\', \'1.2.826.0.1.3680043.8.498.12526876471494728147635430816990398219\', \'1.2.826.0.1.3680043.8.498.92386519544130078776672685639580497728\', \'1.2.826.0.1.3680043.8.498.64421299492271548595446260681698412384\', \'1.2.826.0.1.3680043.8.498.12248983667596865960950671947508678392\', \'1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547\', \'1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791\', \'1.2.826.0.1.3680043.8.498.34740783941206992242367609204835771599\', \'1.2.826.0.1.3680043.8.498.14229598279617168024212382336641104878\', \'1.2.826.0.1.3680043.8.498.92063841381005962285592592672311044745\', \'1.2.826.0.1.3680043.8.498.11385845157446557781575804962818981994\', \'1.2.826.0.1.3680043.8.498.77611073647079414772859496272387571059\', \'1.2.826.0.1.3680043.8.498.35836928928058091851993989259747528067\', \'1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727\', \'1.2.826.0.1.3680043.8.498.64752580249608101443758975861213065048\', \'1.2.826.0.1.3680043.8.498.85499036441349866810703945289793057098\', \'1.2.826.0.1.3680043.8.498.66805139646641294853509958688742554525\', \'1.2.826.0.1.3680043.8.498.92407834172018110142589158281575319154\', \'1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721\', \'1.2.826.0.1.3680043.8.498.11070900122260301212131296598983372089\', \'1.2.826.0.1.3680043.8.498.81351574724227319632426427855692606694\', \'1.2.826.0.1.3680043.8.498.11791567830371683660578052737702299124\', \'1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115\', \'1.2.826.0.1.3680043.8.498.70375834273924102402706635540898552043\', \'1.2.826.0.1.3680043.8.498.80698510014594772882040509716764819716\', \'1.2.826.0.1.3680043.8.498.68168197665046786209369894696801397179\', \'1.2.826.0.1.3680043.8.498.49818874332345742217581267858748819921\', \'1.2.826.0.1.3680043.8.498.11906266535122698149967624908585010734\', \'1.2.826.0.1.3680043.8.498.85748882309364598407865143890978767674\', \'1.2.826.0.1.3680043.8.498.12303161232396569915032951761728884199\', \'1.2.826.0.1.3680043.8.498.39068632185013624181565849012182917752\', \'1.2.826.0.1.3680043.8.498.69311065409015951283858403030798356475\', \'1.2.826.0.1.3680043.8.498.48750912584073496502856009873745362236\', \'1.2.826.0.1.3680043.8.498.61904856234837992279531381374350177464\', \'1.2.826.0.1.3680043.8.498.65530757406333318449497070048719890442\', \'1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736\', \'1.2.826.0.1.3680043.8.498.78646473961447523033955716877524794707\', \'1.2.826.0.1.3680043.8.498.55653770127824574123236841437998533498\', \'1.2.826.0.1.3680043.8.498.81822709772232914298414700148454511661\', \'1.2.826.0.1.3680043.8.498.59426134288436219215525445839849588348\', \'1.2.826.0.1.3680043.8.498.30832771419882354941430609179589560374\', \'1.2.826.0.1.3680043.8.498.72172265826902387784016098389863005666\', \'1.2.826.0.1.3680043.8.498.97797693721516872008302328638374252668\', \'1.2.826.0.1.3680043.8.498.15103255966832759925770334719703425755\', \'1.2.826.0.1.3680043.8.498.45625191292968830335971647388746041117\', \'1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725\', \'1.2.826.0.1.3680043.8.498.87267658933515453963224923798100651549\', \'1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941\', \'1.2.826.0.1.3680043.8.498.58143227319316521737029358930629086887\', \'1.2.826.0.1.3680043.8.498.39468803878805753474151381590569125402\', \'1.2.826.0.1.3680043.8.498.77378803863827312829317008636359414012\', \'1.2.826.0.1.3680043.8.498.18518393141873346710197645305647124682\', \'1.2.826.0.1.3680043.8.498.84123495463663584148839831457665741488\', \'1.2.826.0.1.3680043.8.498.95497572581157858069692837655072209020\', \'1.2.826.0.1.3680043.8.498.72270915206075693056344475438055463113\', \'1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320\', \'1.2.826.0.1.3680043.8.498.45072896942121803896760563154385918032\', \'1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494\', \'1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068\', \'1.2.826.0.1.3680043.8.498.31610133883321894510649078615659708431\', \'1.2.826.0.1.3680043.8.498.13286048014183811835847662577023535662\', \'1.2.826.0.1.3680043.8.498.77735690782343511872544703464260892488\', \'1.2.826.0.1.3680043.8.498.51126807380879212261057967703844943102\', \'1.2.826.0.1.3680043.8.498.88845193477394064535233691626377077328\', \'1.2.826.0.1.3680043.8.498.13203246712933982189680087013566455075\', \'1.2.826.0.1.3680043.8.498.38584486469155131627808759862008577200\', \'1.2.826.0.1.3680043.8.498.98081486779628580946589119341754353354\', \'1.2.826.0.1.3680043.8.498.41640481150213029810772014344448781554\', \'1.2.826.0.1.3680043.8.498.11298099571581989006135838942599921393\', \'1.2.826.0.1.3680043.8.498.33584205642521728950676369503690551550\', \'1.2.826.0.1.3680043.8.498.72653666963736486779382314125782779004\', \'1.2.826.0.1.3680043.8.498.10601062881015669567737602031338578146\', \'1.2.826.0.1.3680043.8.498.10511712617129216774539441890488742560\', \'1.2.826.0.1.3680043.8.498.91431012289716035498822977054491790785\', \'1.2.826.0.1.3680043.8.498.37147839676713817130306478240352601535\', \'1.2.826.0.1.3680043.8.498.74214829456846525506232367308573550186\', \'1.2.826.0.1.3680043.8.498.21322638458116180116030396109651929596\', \'1.2.826.0.1.3680043.8.498.13222515831888962408632292783773537312\', \'1.2.826.0.1.3680043.8.498.95796009319232592137564145023096016872\', \'1.2.826.0.1.3680043.8.498.12982826264059547618124186257414818172\', \'1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910\', \'1.2.826.0.1.3680043.8.498.25679930222284447338484621953345520487\', \'1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852\', \'1.2.826.0.1.3680043.8.498.11314687816975146963598226343179928616\', \'1.2.826.0.1.3680043.8.498.16200518336931321995379999168737787921\', \'1.2.826.0.1.3680043.8.498.11394861774586578112386386342670500749\', \'1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967\', \'1.2.826.0.1.3680043.8.498.94722363227977785887442194094888460046\', \'1.2.826.0.1.3680043.8.498.69033651423155214548081618535669641812\', \'1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946\', \'1.2.826.0.1.3680043.8.498.24588524807799377432185260702152371503\', \'1.2.826.0.1.3680043.8.498.49880369313850883713947031303955494125\', \'1.2.826.0.1.3680043.8.498.11706378290185110677863653788588307917\', \'1.2.826.0.1.3680043.8.498.83030371526667747898709613600825477966\', \'1.2.826.0.1.3680043.8.498.66321903250295261063309947910189882952\', \'1.2.826.0.1.3680043.8.498.10964287649604862436796341368993301245\', \'1.2.826.0.1.3680043.8.498.12158630368692962312115489365152319457\', \'1.2.826.0.1.3680043.8.498.22057596860528658818833903685827585805\', \'1.2.826.0.1.3680043.8.498.80851365368114686105558347899990395309\', \'1.2.826.0.1.3680043.8.498.55300053252365037422745072128951402593\', \'1.2.826.0.1.3680043.8.498.27656830051511853680197276412365020780\', \'1.2.826.0.1.3680043.8.498.13086260989634665810181105278930088749\', \'1.2.826.0.1.3680043.8.498.12189863023888657759738596583562992408\', \'1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121\', \'1.2.826.0.1.3680043.8.498.86020966816754359840982061252012700091\', \'1.2.826.0.1.3680043.8.498.99052905406751383674032587504235261874\', \'1.2.826.0.1.3680043.8.498.60189166269847256202786309096136242746\', \'1.2.826.0.1.3680043.8.498.44321626715626508427287815810252844889\', \'1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754\', \'1.2.826.0.1.3680043.8.498.76167153566894354739332364376483514559\', \'1.2.826.0.1.3680043.8.498.11218782547268547937641080283066300678\', \'1.2.826.0.1.3680043.8.498.82744514729782879778828512295184306538\', \'1.2.826.0.1.3680043.8.498.62400371770962739002393364782172386752\', \'1.2.826.0.1.3680043.8.498.79038153803632889185429988483097161887\', \'1.2.826.0.1.3680043.8.498.12719455674338091089495020475033082405\', \'1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995\', \'1.2.826.0.1.3680043.8.498.32862727544234948830522036721585095753\', \'1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547\', \'1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190\', \'1.2.826.0.1.3680043.8.498.50810596378229719761556926464290468652\', \'1.2.826.0.1.3680043.8.498.54030363830629950591221870797594277092\', \'1.2.826.0.1.3680043.8.498.13136392756016978765204420377500607545\', \'1.2.826.0.1.3680043.8.498.83235374118671754689663732057314083332\', \'1.2.826.0.1.3680043.8.498.63797588614544518196205411126336672334\', \'1.2.826.0.1.3680043.8.498.97055845147740306780908051236353219644\', \'1.2.826.0.1.3680043.8.498.97303893645051596044169857585815133097\', \'1.2.826.0.1.3680043.8.498.90433488790413321102007716267340900056\', \'1.2.826.0.1.3680043.8.498.69846596572083911408018130524077120494\', \'1.2.826.0.1.3680043.8.498.45326662718293919909188368237039419949\', \'1.2.826.0.1.3680043.8.498.88074618626471004030644935418777151905\', \'1.2.826.0.1.3680043.8.498.39394842976048577715164357752399606924\', \'1.2.826.0.1.3680043.8.498.37057548998853865829402084168010676727\', \'1.2.826.0.1.3680043.8.498.51917043703891091519740084517167861965\', \'1.2.826.0.1.3680043.8.498.10287488031206457680758379452936914669\', \'1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655\', \'1.2.826.0.1.3680043.8.498.11495133447993934376048707763538781434\', \'1.2.826.0.1.3680043.8.498.78188863861812651079751164964756388411\', \'1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443\', \'1.2.826.0.1.3680043.8.498.34636178772986348716257313987840123906\', \'1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551\', \'1.2.826.0.1.3680043.8.498.26542435427465415008860869796504112727\', \'1.2.826.0.1.3680043.8.498.87998832540756872371878613339628615880\', \'1.2.826.0.1.3680043.8.498.44941174190375343987327566550204051010\', \'1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500\', \'1.2.826.0.1.3680043.8.498.74052473719853343027548507306186291134\', \'1.2.826.0.1.3680043.8.498.13024625496981667037628275235696732200\', \'1.2.826.0.1.3680043.8.498.67613388743126519966672593774754849330\', \'1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918\', \'1.2.826.0.1.3680043.8.498.81024897463656728358703040759074583549\', \'1.2.826.0.1.3680043.8.498.94601583997736688762302974573903982304\', \'1.2.826.0.1.3680043.8.498.11159546221032123495135103648570690656\', \'1.2.826.0.1.3680043.8.498.12700981063762577397919340901063297803\', \'1.2.826.0.1.3680043.8.498.47666607805434428477774854287399691400\', \'1.2.826.0.1.3680043.8.498.20301796413163444642416918108994411350\', \'1.2.826.0.1.3680043.8.498.67157948144868269956661697924341234924\', \'1.2.826.0.1.3680043.8.498.92722393862174380736027353110424417588\', \'1.2.826.0.1.3680043.8.498.75384600248630012126841890313149674691\', \'1.2.826.0.1.3680043.8.498.42778524243264061817587398263842942394\', \'1.2.826.0.1.3680043.8.498.81797437102217585216278133168092577952\', \'1.2.826.0.1.3680043.8.498.11595338216297251382188195835520492789\', \'1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536\', \'1.2.826.0.1.3680043.8.498.11067200656721609390997819562401839124\', \'1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290\', \'1.2.826.0.1.3680043.8.498.10349368332722015923160611226877741306\', \'1.2.826.0.1.3680043.8.498.68166103964518457268906879600985959436\', \'1.2.826.0.1.3680043.8.498.94753144127920558221268610279343359449\', \'1.2.826.0.1.3680043.8.498.10673300644364559354643106316993112731\', \'1.2.826.0.1.3680043.8.498.10256441455107860474283716992657597873\', \'1.2.826.0.1.3680043.8.498.12765434812409174436720333864278447770\', \'1.2.826.0.1.3680043.8.498.12100254887364913022980951672127944515\', \'1.2.826.0.1.3680043.8.498.28360174615866226538417948042981668998\', \'1.2.826.0.1.3680043.8.498.50969691234820675847736765735133138324\', \'1.2.826.0.1.3680043.8.498.11923218013055841952315996726408812182\', \'1.2.826.0.1.3680043.8.498.48829151552094424673693975665028690142\', \'1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465\', \'1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886\', \'1.2.826.0.1.3680043.8.498.12285469859900617016859868410323043734\', \'1.2.826.0.1.3680043.8.498.12699751030439008569144814819343161608\', \'1.2.826.0.1.3680043.8.498.75168480371573015758963049782687952716\', \'1.2.826.0.1.3680043.8.498.78003548255041391467292087330671945247\', \'1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699\', \'1.2.826.0.1.3680043.8.498.11878847808072119549790690649256090471\', \'1.2.826.0.1.3680043.8.498.90858397443821905121918628339486070979\', \'1.2.826.0.1.3680043.8.498.97464332941382646555656038793589898504\', \'1.2.826.0.1.3680043.8.498.12107680313133383587522762626929218051\', \'1.2.826.0.1.3680043.8.498.29443692810539372078922146461173545451\', \'1.2.826.0.1.3680043.8.498.22596004572382994883130705870731193046\', \'1.2.826.0.1.3680043.8.498.13291943370279119939499960846843879156\', \'1.2.826.0.1.3680043.8.498.33387430823462011959198830568803117633\', \'1.2.826.0.1.3680043.8.498.82548251499640077572026404183402934330\', \'1.2.826.0.1.3680043.8.498.33393833442961030521286257650328223086\', \'1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004\', \'1.2.826.0.1.3680043.8.498.87680825247567493080163504511649789348\', \'1.2.826.0.1.3680043.8.498.25604547074218147115077644853162295379\', \'1.2.826.0.1.3680043.8.498.13246496699722524712272293738359002501\', \'1.2.826.0.1.3680043.8.498.79565393323323007178523850093527250842\', \'1.2.826.0.1.3680043.8.498.53262958748075737242820629127001253146\', \'1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762\', \'1.2.826.0.1.3680043.8.498.90551187439998209641370107095424494085\', \'1.2.826.0.1.3680043.8.498.85728170022176096748897349447671687164\', \'1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870\', \'1.2.826.0.1.3680043.8.498.80275294885165896124489476449616436102\', \'1.2.826.0.1.3680043.8.498.12150502202634860887314001818431257267\', \'1.2.826.0.1.3680043.8.498.12238865510201989224648662550299925120\', \'1.2.826.0.1.3680043.8.498.23427198594727947883953289189443006288\', \'1.2.826.0.1.3680043.8.498.28481442608487320644829189127206806471\', \'1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395\', \'1.2.826.0.1.3680043.8.498.37633035024065331400434438202028794597\', \'1.2.826.0.1.3680043.8.498.12130868132537772155475981935868588761\', \'1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585\', \'1.2.826.0.1.3680043.8.498.20204907530808218314127264658715054041\', \'1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727\', \'1.2.826.0.1.3680043.8.498.12322760603170246518010233325038530097\', \'1.2.826.0.1.3680043.8.498.22053727963380087706894183799325067544\', \'1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953\', \'1.2.826.0.1.3680043.8.498.11515987024418648685676369132061127664\', \'1.2.826.0.1.3680043.8.498.64950600385134699009390197520158979183\', \'1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751\', \'1.2.826.0.1.3680043.8.498.63216944317982327305447295456609309952\', \'1.2.826.0.1.3680043.8.498.42816339547796375466199938922141987775\', \'1.2.826.0.1.3680043.8.498.20472525526345683475357475395677168372\', \'1.2.826.0.1.3680043.8.498.25766387275525303860953321689718246405\', \'1.2.826.0.1.3680043.8.498.85932495800565018166769303205378781707\', \'1.2.826.0.1.3680043.8.498.80325205657974094483016625702390834774\', \'1.2.826.0.1.3680043.8.498.55065105486648830377200570585789726800\', \'1.2.826.0.1.3680043.8.498.20266523169651039493799975212435349766\', \'1.2.826.0.1.3680043.8.498.13146466304138609448483938844487385775\', \'1.2.826.0.1.3680043.8.498.97705920250069816684861966612302390614\', \'1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664\', \'1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938\', \'1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334\', \'1.2.826.0.1.3680043.8.498.28746954364434312909368235768936980607\', \'1.2.826.0.1.3680043.8.498.11867198902144602395671904591496233129\', \'1.2.826.0.1.3680043.8.498.28634248756653012357434232845788268719\', \'1.2.826.0.1.3680043.8.498.50487158921933419035809338652990508020\', \'1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911\', \'1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365\', \'1.2.826.0.1.3680043.8.498.13024250150960117282590397343184827466\', \'1.2.826.0.1.3680043.8.498.30756534269149851281518502475458067520\', \'1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350\', \'1.2.826.0.1.3680043.8.498.96094001324999448553107805455529187871\', \'1.2.826.0.1.3680043.8.498.36180248571044206939894546783755557323\', \'1.2.826.0.1.3680043.8.498.56880070152758706555016296832842012883\', \'1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868\', \'1.2.826.0.1.3680043.8.498.15630146025588580188363070220779173742\', \'1.2.826.0.1.3680043.8.498.13207489724395276840990928372610055495\', \'1.2.826.0.1.3680043.8.498.27184343589256623289925252380526385858\', \'1.2.826.0.1.3680043.8.498.83121378128521985713113311344685112825\', \'1.2.826.0.1.3680043.8.498.11561519302314146020052258325459371962\', \'1.2.826.0.1.3680043.8.498.87769640167791104899516835133527936885\', \'1.2.826.0.1.3680043.8.498.20640858366001462094044810443654811785\', \'1.2.826.0.1.3680043.8.498.72931900540220826595003388066347326492\', \'1.2.826.0.1.3680043.8.498.82169173362972793731001247413129649263\', \'1.2.826.0.1.3680043.8.498.69397705155385306818511499752200941968\', \'1.2.826.0.1.3680043.8.498.36930256628357053046574486650916442047\', \'1.2.826.0.1.3680043.8.498.10726624821670417064567481545750543282\', \'1.2.826.0.1.3680043.8.498.64022949342538714157641824612944253302\', \'1.2.826.0.1.3680043.8.498.63279205896049798083339413931561428540\', \'1.2.826.0.1.3680043.8.498.24758830794084473905762034032291806038\', \'1.2.826.0.1.3680043.8.498.88211355128040927831407007101167868685\', \'1.2.826.0.1.3680043.8.498.37030593568962187244853961673610373876\', \'1.2.826.0.1.3680043.8.498.73539750992809874106013052513162732951\', \'1.2.826.0.1.3680043.8.498.52639459702574533721011212286428487650\', \'1.2.826.0.1.3680043.8.498.70507536279875475164400849670959748314\', \'1.2.826.0.1.3680043.8.498.72237065473467452226073540784070938342\', \'1.2.826.0.1.3680043.8.498.10731612202208916303411280624156291171\', \'1.2.826.0.1.3680043.8.498.10386894500329189457070808193345170500\', \'1.2.826.0.1.3680043.8.498.90159195278632080234412316065469499332\', \'1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104\', \'1.2.826.0.1.3680043.8.498.77823946901144772346251781172043480398\', \'1.2.826.0.1.3680043.8.498.90342120331229511020327052213821820086\', \'1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933\', \'1.2.826.0.1.3680043.8.498.12491395125580387847235487841392723348\', \'1.2.826.0.1.3680043.8.498.16666020201961191623151386398626511849\', \'1.2.826.0.1.3680043.8.498.10736346409444614168571281812134757732\', \'1.2.826.0.1.3680043.8.498.12229628534316014336864328244217875465\', \'1.2.826.0.1.3680043.8.498.73682608760611536443754406204042210115\', \'1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268\', \'1.2.826.0.1.3680043.8.498.53963759852017695930808272438606800422\', \'1.2.826.0.1.3680043.8.498.58768083117076186082335984587845964130\', \'1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229\', \'1.2.826.0.1.3680043.8.498.12957752106697316015996777476239762111\', \'1.2.826.0.1.3680043.8.498.75420186662442688955616630735303321846\', \'1.2.826.0.1.3680043.8.498.23677916081296740815989149501889343663\', \'1.2.826.0.1.3680043.8.498.44476475746458439091354236934188679569\', \'1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871\', \'1.2.826.0.1.3680043.8.498.11752091432368750506312921028559788225\', \'1.2.826.0.1.3680043.8.498.12632129640405661298623288977632682289\', \'1.2.826.0.1.3680043.8.498.27245171350765293537022546751529180545\', \'1.2.826.0.1.3680043.8.498.19887850741193535685529812263489892001\', \'1.2.826.0.1.3680043.8.498.12769221625850663248477857087347890908\', \'1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234\', \'1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029\', \'1.2.826.0.1.3680043.8.498.12619895582462244920124420363095282842\', \'1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626\', \'1.2.826.0.1.3680043.8.498.15065556607080211420503043544482923211\', \'1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002\', \'1.2.826.0.1.3680043.8.498.12815132582382825980088023336277934996\', \'1.2.826.0.1.3680043.8.498.31282366761674650660824706600044115805\', \'1.2.826.0.1.3680043.8.498.30206987686375669521451035370290737934\', \'1.2.826.0.1.3680043.8.498.78972123731732056524948076388825106061\', \'1.2.826.0.1.3680043.8.498.14958560978176121375035839707004667832\', \'1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343\', \'1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310\', \'1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768\', \'1.2.826.0.1.3680043.8.498.32479930837336533083798155873656743007\', \'1.2.826.0.1.3680043.8.498.12501925425699863373513819814520995919\', \'1.2.826.0.1.3680043.8.498.19706775737718777290660344285046025310\', \'1.2.826.0.1.3680043.8.498.12637591861595747322913829625646935491\', \'1.2.826.0.1.3680043.8.498.35971292331621735865925455612813197566\', \'1.2.826.0.1.3680043.8.498.53528011450161116437223332448942321596\', \'1.2.826.0.1.3680043.8.498.13152690996267297597178587721720788387\', \'1.2.826.0.1.3680043.8.498.11221318524515817857172161854667955115\', \'1.2.826.0.1.3680043.8.498.12267944258766605501055560486655358039\', \'1.2.826.0.1.3680043.8.498.10405028141768633100332386157519916334\', \'1.2.826.0.1.3680043.8.498.31171050331393253655760166990008263793\', \'1.2.826.0.1.3680043.8.498.72182847112910012489603688436912469641\', \'1.2.826.0.1.3680043.8.498.28237068093947300427040925444622840795\', \'1.2.826.0.1.3680043.8.498.90466423642678253933409966898120881010\', \'1.2.826.0.1.3680043.8.498.93141871723058796160969424330424882014\', \'1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547\', \'1.2.826.0.1.3680043.8.498.42389063782063360163798575455522539633\', \'1.2.826.0.1.3680043.8.498.22249763791584161324425922920269052844\', \'1.2.826.0.1.3680043.8.498.98039072306618106705944452906395982381\', \'1.2.826.0.1.3680043.8.498.37572710131396388050286115883363501023\', \'1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816\', \'1.2.826.0.1.3680043.8.498.10627795990821506536342369227971125477\', \'1.2.826.0.1.3680043.8.498.68919687595587004665658610968739852899\', \'1.2.826.0.1.3680043.8.498.38701927697650476092773702704871872598\', \'1.2.826.0.1.3680043.8.498.12208986088767216690584621773389997968\', \'1.2.826.0.1.3680043.8.498.59646951619367295963072805933618947027\', \'1.2.826.0.1.3680043.8.498.12219658385745696203053079500246399481\', \'1.2.826.0.1.3680043.8.498.27161418589407291713644384596541786246\', \'1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581\', \'1.2.826.0.1.3680043.8.498.13353249891620291438488727876474802879\', \'1.2.826.0.1.3680043.8.498.18625377919309205857120774497713011583\', \'1.2.826.0.1.3680043.8.498.85883562680953255833567595509408888410\', \'1.2.826.0.1.3680043.8.498.23378338408981699466914366379590099738\', \'1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537\', \'1.2.826.0.1.3680043.8.498.76093766204928582819565636658977147523\', \'1.2.826.0.1.3680043.8.498.89026102874017923932934673207699042800\', \'1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878\', \'1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542\', \'1.2.826.0.1.3680043.8.498.77508762938574470575737016745357126587\', \'1.2.826.0.1.3680043.8.498.12785792890626338008419452097492192088\', \'1.2.826.0.1.3680043.8.498.82523073559792523830827952754346276051\', \'1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260\', \'1.2.826.0.1.3680043.8.498.74952769952682175467667812958859010329\', \'1.2.826.0.1.3680043.8.498.10777331080659372520952742261140923555\', \'1.2.826.0.1.3680043.8.498.53370670740736715669115043403494610200\', \'1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830\', \'1.2.826.0.1.3680043.8.498.84062980592611347665739105270731842220\', \'1.2.826.0.1.3680043.8.498.44441813293962885868294317830142850713\', \'1.2.826.0.1.3680043.8.498.87645071186049036230936726742129446188\', \'1.2.826.0.1.3680043.8.498.11201589885283493748455257854104771618\', \'1.2.826.0.1.3680043.8.498.50707407856055788475528375847872674241\', \'1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991\', \'1.2.826.0.1.3680043.8.498.35991881811942962264382690224767291006\', \'1.2.826.0.1.3680043.8.498.38303182087206935163155565441234078389\', \'1.2.826.0.1.3680043.8.498.94677297888135961827159679484207792618\', \'1.2.826.0.1.3680043.8.498.46036209147177723970966724006242055397\', \'1.2.826.0.1.3680043.8.498.64022112049842890696609934970222024490\', \'1.2.826.0.1.3680043.8.498.80161127293548940216292842707683784128\', \'1.2.826.0.1.3680043.8.498.65312506382127486486519184894274326380\', \'1.2.826.0.1.3680043.8.498.14188540532926067083914263496492255704\', \'1.2.826.0.1.3680043.8.498.76950154099427419631807259601983648254\', \'1.2.826.0.1.3680043.8.498.46228189366201925238527911905072482095\', \'1.2.826.0.1.3680043.8.498.95560861705581165570145459410262387310\', \'1.2.826.0.1.3680043.8.498.13026463185033527579856819607371375594\', \'1.2.826.0.1.3680043.8.498.13295633458001436592005068637719195786\', \'1.2.826.0.1.3680043.8.498.74362364640190722036251935797260325860\', \'1.2.826.0.1.3680043.8.498.13093528938282977263747792690249036456\', \'1.2.826.0.1.3680043.8.498.12819437506700585448407291050085063174\', \'1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669\', \'1.2.826.0.1.3680043.8.498.13139684164906828131777150061532685584\', \'1.2.826.0.1.3680043.8.498.36035264348264378094569428634302238394\', \'1.2.826.0.1.3680043.8.498.91497176426480836856097456438014037999\', \'1.2.826.0.1.3680043.8.498.15399233309269178192583032682669258885\', \'1.2.826.0.1.3680043.8.498.82012532783011843175172575198326783678\', \'1.2.826.0.1.3680043.8.498.12775050278338123442507219047972133270\', \'1.2.826.0.1.3680043.8.498.10492484093480313596330475837639172625\', \'1.2.826.0.1.3680043.8.498.25505153402978560400776672230220616863\', \'1.2.826.0.1.3680043.8.498.11448865770537436004548718998804636234\', \'1.2.826.0.1.3680043.8.498.72191353854496806844040272237700346774\', \'1.2.826.0.1.3680043.8.498.10351168527688244230248760520733020605\', \'1.2.826.0.1.3680043.8.498.13262053361139017457941697384315610025\', \'1.2.826.0.1.3680043.8.498.99413398303724030328059886749330413303\', \'1.2.826.0.1.3680043.8.498.86315835011470465506843590014408831373\', \'1.2.826.0.1.3680043.8.498.73231153161312542696053060380619432126\', \'1.2.826.0.1.3680043.8.498.37558426555271635608942179071280432530\', \'1.2.826.0.1.3680043.8.498.12168045829469202472124733625621943469\', \'1.2.826.0.1.3680043.8.498.56497268548664917337130798701015887461\', \'1.2.826.0.1.3680043.8.498.13165770394160595162984722917452661205\', \'1.2.826.0.1.3680043.8.498.63002559681883387414876870926128159682\', \'1.2.826.0.1.3680043.8.498.36645945703275956149107297016813148264\', \'1.2.826.0.1.3680043.8.498.15444316810805746453392291780593934328\', \'1.2.826.0.1.3680043.8.498.12630423349626505319792444053874011718\', \'1.2.826.0.1.3680043.8.498.12785377697188564760194569072434828180\', \'1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870\', \'1.2.826.0.1.3680043.8.498.69495014653333523580255444915082056825\', \'1.2.826.0.1.3680043.8.498.89987866402625167201267936974120824712\', \'1.2.826.0.1.3680043.8.498.58482152742263188503728990365891896621\', \'1.2.826.0.1.3680043.8.498.45059917085750212913588683242522799083\', \'1.2.826.0.1.3680043.8.498.22426390571898968732573184721547490379\', \'1.2.826.0.1.3680043.8.498.62760659205261492447481830774781950875\', \'1.2.826.0.1.3680043.8.498.46700895961751940291334058396794206479\', \'1.2.826.0.1.3680043.8.498.21798342483930597563659425189400765869\', \'1.2.826.0.1.3680043.8.498.94117540970936035429846288243624086693\', \'1.2.826.0.1.3680043.8.498.91499489440453244654238202136675083579\', \'1.2.826.0.1.3680043.8.498.46305694398420300356121078124030126469\', \'1.2.826.0.1.3680043.8.498.10756245293186761042431460535210737327\', \'1.2.826.0.1.3680043.8.498.11964919539297578836040550782883323317\', \'1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578\', \'1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950\', \'1.2.826.0.1.3680043.8.498.98102475310924339032647376491442498974\', \'1.2.826.0.1.3680043.8.498.11291875173369366854889507729635968509\', \'1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504\', \'1.2.826.0.1.3680043.8.498.63204735160524958513470115761010950181\', \'1.2.826.0.1.3680043.8.498.28848511016500781798837621261660844237\', \'1.2.826.0.1.3680043.8.498.26373657290121957625263322898744712730\', \'1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205\', \'1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645\', \'1.2.826.0.1.3680043.8.498.11897083656099489353496670273111097794\', \'1.2.826.0.1.3680043.8.498.97689748704772735695349883980413529091\', \'1.2.826.0.1.3680043.8.498.64946212153370539647301308676689632525\', \'1.2.826.0.1.3680043.8.498.99237666579371266866432031468578336182\', \'1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678\', \'1.2.826.0.1.3680043.8.498.71707317731065645278250956442336515504\', \'1.2.826.0.1.3680043.8.498.69470130842235635628917024269527211321\', \'1.2.826.0.1.3680043.8.498.12748840878706905309050091576449488666\', \'1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341\', \'1.2.826.0.1.3680043.8.498.12176469158911896593955662590695392044\', \'1.2.826.0.1.3680043.8.498.58522891982092784622515590012748600644\', \'1.2.826.0.1.3680043.8.498.45392604471538776666611947492530314159\', \'1.2.826.0.1.3680043.8.498.60347768295306794807599672836772410567\', \'1.2.826.0.1.3680043.8.498.46715670142711601593792518196789347552\', \'1.2.826.0.1.3680043.8.498.21605506484276750075895228202564072044\', \'1.2.826.0.1.3680043.8.498.11659861228342609979638095779959188093\', \'1.2.826.0.1.3680043.8.498.67490059263503953166467514121525025332\', \'1.2.826.0.1.3680043.8.498.82757616216788919181323176147396770758\', \'1.2.826.0.1.3680043.8.498.84047134932018909998648460546980598926\', \'1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290\', \'1.2.826.0.1.3680043.8.498.10183939841890895037058579455663880292\', \'1.2.826.0.1.3680043.8.498.45454539422542821586939369440135728144\', \'1.2.826.0.1.3680043.8.498.40041164441539690764994039690598305675\', \'1.2.826.0.1.3680043.8.498.17427902052389033416295416299697276519\', \'1.2.826.0.1.3680043.8.498.34360298211002490108642003575962873984\', \'1.2.826.0.1.3680043.8.498.94635110520016467882028406590905441740\', \'1.2.826.0.1.3680043.8.498.63151733210715530368323422018984705316\', \'1.2.826.0.1.3680043.8.498.11190292536746235063218740991532759863\', \'1.2.826.0.1.3680043.8.498.53893586200869613140666806653041177558\', \'1.2.826.0.1.3680043.8.498.64240637244650029205175793729578897734\', \'1.2.826.0.1.3680043.8.498.57160448024145680660335079050450776852\', \'1.2.826.0.1.3680043.8.498.97020520267554584684160936968569132254\', \'1.2.826.0.1.3680043.8.498.74127742661271965093205833471645893520\', \'1.2.826.0.1.3680043.8.498.11386074316342852649653530230563922899\', \'1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386\', \'1.2.826.0.1.3680043.8.498.11501070108039994411346060488666341726\', \'1.2.826.0.1.3680043.8.498.95223549428749744711279985897241419512\', \'1.2.826.0.1.3680043.8.498.11957476321850340968244508976719889277\', \'1.2.826.0.1.3680043.8.498.98255454187594895604610283922744869720\', \'1.2.826.0.1.3680043.8.498.87102698749288269439517333660896096520\', \'1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056\', \'1.2.826.0.1.3680043.8.498.24068239310155798723127213793623172530\', \'1.2.826.0.1.3680043.8.498.95464807346156547985763374990661923815\', \'1.2.826.0.1.3680043.8.498.40386781168330493993852288092654487818\', \'1.2.826.0.1.3680043.8.498.21312254534068992054745674608909722454\', \'1.2.826.0.1.3680043.8.498.68799355386870083063665227055340582612\', \'1.2.826.0.1.3680043.8.498.11264140845869814478794346490273061499\', \'1.2.826.0.1.3680043.8.498.42221373360893091454504422964928099406\', \'1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593\', \'1.2.826.0.1.3680043.8.498.62983291359021407345388012674614537807\', \'1.2.826.0.1.3680043.8.498.10371162371589200772388569623043753617\', \'1.2.826.0.1.3680043.8.498.11487799220518102363643849548556099719\', \'1.2.826.0.1.3680043.8.498.59420153636832938628088507776889305823\', \'1.2.826.0.1.3680043.8.498.10243300429607935614655219482457289307\', \'1.2.826.0.1.3680043.8.498.20156361590688072955579948030116579099\', \'1.2.826.0.1.3680043.8.498.35777075794856195065665011443485613681\', \'1.2.826.0.1.3680043.8.498.68158735450883179427440578190038002606\', \'1.2.826.0.1.3680043.8.498.76772148369531098768751348676329945789\', \'1.2.826.0.1.3680043.8.498.12509825237087140646629745447753158673\', \'1.2.826.0.1.3680043.8.498.91230059913252604666971822137553690811\', \'1.2.826.0.1.3680043.8.498.31099878224304681441631176766255029271\', \'1.2.826.0.1.3680043.8.498.75282577107802752393765861202996650638\', \'1.2.826.0.1.3680043.8.498.90193654806390401129887447441469319421\', \'1.2.826.0.1.3680043.8.498.13975027824128345697756313869165838055\', \'1.2.826.0.1.3680043.8.498.11851285389689341495961233129719325417\', \'1.2.826.0.1.3680043.8.498.10676253869668012791900504544106495589\', \'1.2.826.0.1.3680043.8.498.67620512607520186685421323991166816451\', \'1.2.826.0.1.3680043.8.498.46884541791777826238103426426267571657\', \'1.2.826.0.1.3680043.8.498.46452612704507986055186132736451572677\', \'1.2.826.0.1.3680043.8.498.35892665011493087075493127252825546314\', \'1.2.826.0.1.3680043.8.498.10534260101860670685224894870019533364\', \'1.2.826.0.1.3680043.8.498.48417903971145834309474934717837250776\', \'1.2.826.0.1.3680043.8.498.81692996137620934043276253858381812472\', \'1.2.826.0.1.3680043.8.498.11467955500991327547315970867313333592\', \'1.2.826.0.1.3680043.8.498.77018355356797290743788712046815540223\', \'1.2.826.0.1.3680043.8.498.32742383730417896527915284404039812924\', \'1.2.826.0.1.3680043.8.498.67456402757136973484427155324664966595\', \'1.2.826.0.1.3680043.8.498.82272793832748838882752586678639564181\', \'1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644\', \'1.2.826.0.1.3680043.8.498.10684020894033412667820478260168097071\', \'1.2.826.0.1.3680043.8.498.12377852033407536077897684568488365785\', \'1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766\', \'1.2.826.0.1.3680043.8.498.89576499383950909889665633610791093334\', \'1.2.826.0.1.3680043.8.498.11726452645519338341002251323463736083\', \'1.2.826.0.1.3680043.8.498.28295265939946677460838974253299313798\', \'1.2.826.0.1.3680043.8.498.12929915324982999692857962099866616265\', \'1.2.826.0.1.3680043.8.498.53745429232602455103594227782517963027\', \'1.2.826.0.1.3680043.8.498.31482725509752510642857072190147133461\', \'1.2.826.0.1.3680043.8.498.97600121574703193898619052529830275028\', \'1.2.826.0.1.3680043.8.498.13027502247317650159940595684214701589\', \'1.2.826.0.1.3680043.8.498.13843368750899904654034226115826130084\', \'1.2.826.0.1.3680043.8.498.21904421326144238681996713485932410596\', \'1.2.826.0.1.3680043.8.498.57845032486422442404222046144765204905\', \'1.2.826.0.1.3680043.8.498.22398731807962276404889164827238996005\', \'1.2.826.0.1.3680043.8.498.12154188739734505793699974626991872621\', \'1.2.826.0.1.3680043.8.498.11596665780617412984766822528862741584\', \'1.2.826.0.1.3680043.8.498.97830923192341890652952842494018952920\', \'1.2.826.0.1.3680043.8.498.11980805687949950504930975019161587970\', \'1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166\', \'1.2.826.0.1.3680043.8.498.57615268093937994500995086275007694564\', \'1.2.826.0.1.3680043.8.498.11646465814751330102194657154003685957\', \'1.2.826.0.1.3680043.8.498.97004483591331502283638578516018997228\', \'1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062\', \'1.2.826.0.1.3680043.8.498.13131420451380633781538717117200087258\', \'1.2.826.0.1.3680043.8.498.44879493336213912007673875382158420811\', \'1.2.826.0.1.3680043.8.498.51427159850769972151411336025891779423\', \'1.2.826.0.1.3680043.8.498.23703817737293327679300330167234268929\', \'1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052\', \'1.2.826.0.1.3680043.8.498.33885566709305241744786004177260915650\', \'1.2.826.0.1.3680043.8.498.36470828078108352859868482351276699300\', \'1.2.826.0.1.3680043.8.498.12689874339887047433232628649886917274\', \'1.2.826.0.1.3680043.8.498.71084400653875583977681616665093993652\', \'1.2.826.0.1.3680043.8.498.34546106114963066554367298449299214677\', \'1.2.826.0.1.3680043.8.498.65834774574423727070722226151408235786\', \'1.2.826.0.1.3680043.8.498.39025433784711757736168236829157583043\', \'1.2.826.0.1.3680043.8.498.56188634971310070481285525830793231849\', \'1.2.826.0.1.3680043.8.498.52745530063404662086111315503240395535\', \'1.2.826.0.1.3680043.8.498.45894475516096841241467193775629362381\', \'1.2.826.0.1.3680043.8.498.21024377993947321291280779867493407199\', \'1.2.826.0.1.3680043.8.498.45862157701704991290669186265816463426\', \'1.2.826.0.1.3680043.8.498.19788612491138642031531719942303619648\', \'1.2.826.0.1.3680043.8.498.86409878732405036646840908998274047391\', \'1.2.826.0.1.3680043.8.498.86974853023888933977090539174537083409\', \'1.2.826.0.1.3680043.8.498.51970198810758997428998886534742243215\', \'1.2.826.0.1.3680043.8.498.61148667330113929729568621580194520169\', \'1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758\', \'1.2.826.0.1.3680043.8.498.11679981444785137570894759919380932611\', \'1.2.826.0.1.3680043.8.498.84827027113422423894439539536275453132\', \'1.2.826.0.1.3680043.8.498.73511703934549662392765328175356230615\', \'1.2.826.0.1.3680043.8.498.26201626427712146573573952548179938304\', \'1.2.826.0.1.3680043.8.498.62154969360158054310282857093665911261\', \'1.2.826.0.1.3680043.8.498.28132506470634542229651932141193521661\', \'1.2.826.0.1.3680043.8.498.14959732862523218997149041503831001633\', \'1.2.826.0.1.3680043.8.498.25765711614507755527825654507920391096\', \'1.2.826.0.1.3680043.8.498.21389336408905524316915532623385847283\', \'1.2.826.0.1.3680043.8.498.82548952153746644951736669119607083230\', \'1.2.826.0.1.3680043.8.498.72395496925770440630088720587484704971\', \'1.2.826.0.1.3680043.8.498.11414215447313658600502835138277811260\', \'1.2.826.0.1.3680043.8.498.55344075316305815718253234447285669886\', \'1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603\', \'1.2.826.0.1.3680043.8.498.60614975796208617794180432816390721331\', \'1.2.826.0.1.3680043.8.498.74243564762473331581882860440560337725\', \'1.2.826.0.1.3680043.8.498.21060274255072711241653174133146945101\', \'1.2.826.0.1.3680043.8.498.17618734998753079190687719289815512319\', \'1.2.826.0.1.3680043.8.498.12831769902826114242555323443809541052\', \'1.2.826.0.1.3680043.8.498.11369844885381156110080055326139875897\', \'1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441\', \'1.2.826.0.1.3680043.8.498.78968420394708302953391711477016139949\', \'1.2.826.0.1.3680043.8.498.86516555757334047240474191101770683543\', \'1.2.826.0.1.3680043.8.498.95321750682026667237717880394536750835\', \'1.2.826.0.1.3680043.8.498.53271710112621372496324855809391580130\', \'1.2.826.0.1.3680043.8.498.11504079342934513987748646905909982588\', \'1.2.826.0.1.3680043.8.498.12207279366369605175255203969558548753\', \'1.2.826.0.1.3680043.8.498.81892104897856100357169774214718573167\', \'1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268\', \'1.2.826.0.1.3680043.8.498.60840220723824894619895264673401129352\', \'1.2.826.0.1.3680043.8.498.11814923363609140180293604689814785969\', \'1.2.826.0.1.3680043.8.498.35965770865730278787229040370963322662\', \'1.2.826.0.1.3680043.8.498.12606136253498337506682547306933254691\', \'1.2.826.0.1.3680043.8.498.56575942633137496743464648679775042979\', \'1.2.826.0.1.3680043.8.498.11569355546217846610131708883985793793\', \'1.2.826.0.1.3680043.8.498.23107903961514228607587963030481481744\', \'1.2.826.0.1.3680043.8.498.86142930956005716169649168306759482915\', \'1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935\', \'1.2.826.0.1.3680043.8.498.10318366890716843554246682186865048846\', \'1.2.826.0.1.3680043.8.498.37332768072749723117839591872457536315\', \'1.2.826.0.1.3680043.8.498.11979906027101464432999416052202616662\', \'1.2.826.0.1.3680043.8.498.12414869188691502885668778427120766321\', \'1.2.826.0.1.3680043.8.498.81475058647553697318837239968139077272\', \'1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265\', \'1.2.826.0.1.3680043.8.498.14679140388328648617386311272361396555\', \'1.2.826.0.1.3680043.8.498.18199465156572252531439880765733639584\', \'1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724\', \'1.2.826.0.1.3680043.8.498.42287188181305579531212818123200473947\', \'1.2.826.0.1.3680043.8.498.17049315826418217889213226790962592213\', \'1.2.826.0.1.3680043.8.498.39547752108509751520306190293259313911\', \'1.2.826.0.1.3680043.8.498.38272345519163551901865251730721530169\', \'1.2.826.0.1.3680043.8.498.11149530279799223497509928655010867960\', \'1.2.826.0.1.3680043.8.498.12098718763390783455090805975786780205\', \'1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538\', \'1.2.826.0.1.3680043.8.498.11010004366885474301784150582039137111\', \'1.2.826.0.1.3680043.8.498.63521033783841675104013677430088400343\', \'1.2.826.0.1.3680043.8.498.13125910783461518266627225743585018143\', \'1.2.826.0.1.3680043.8.498.40837296112545776586904280088902000849\', \'1.2.826.0.1.3680043.8.498.77498121906981512168120474112422492971\', \'1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136\', \'1.2.826.0.1.3680043.8.498.30978547345367837000803077743620171961\', \'1.2.826.0.1.3680043.8.498.81790990482142045640192729399175106255\', \'1.2.826.0.1.3680043.8.498.65356758967883114612797250487949537460\', \'1.2.826.0.1.3680043.8.498.11223691171233118900160265287691267884\', \'1.2.826.0.1.3680043.8.498.55887747667305814945719986974538814207\', \'1.2.826.0.1.3680043.8.498.12331841096593105748526634055968654914\', \'1.2.826.0.1.3680043.8.498.63565089119380386830816381560603756258\', \'1.2.826.0.1.3680043.8.498.74140806279409854828032496578072554031\', \'1.2.826.0.1.3680043.8.498.11868154141292577776796381772987549697\', \'1.2.826.0.1.3680043.8.498.12577828789206467631166875572423539298\', \'1.2.826.0.1.3680043.8.498.11954711639198972958656966617027374769\', \'1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878\', \'1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382\', \'1.2.826.0.1.3680043.8.498.11596866241403598097171345671773992202\', \'1.2.826.0.1.3680043.8.498.29055892482211392922525222851162308510\', \'1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363\', \'1.2.826.0.1.3680043.8.498.45463321406810606039579747001339703552\', \'1.2.826.0.1.3680043.8.498.30381005430261460472880465705496199540\', \'1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414\', \'1.2.826.0.1.3680043.8.498.31806829168516955528191705338336428721\', \'1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590\', \'1.2.826.0.1.3680043.8.498.27934397251575987048967084987994202973\', \'1.2.826.0.1.3680043.8.498.65515828911930515002046151253057175178\', \'1.2.826.0.1.3680043.8.498.64408775259838960958354835366765941676\', \'1.2.826.0.1.3680043.8.498.11798183157109435753679723596499976762\', \'1.2.826.0.1.3680043.8.498.22263857526032448247006468310072414129\', \'1.2.826.0.1.3680043.8.498.38420384692181444061893234153745102059\', \'1.2.826.0.1.3680043.8.498.90679199017361780574384378857945322240\', \'1.2.826.0.1.3680043.8.498.33284228284209148349668722931307588402\', \'1.2.826.0.1.3680043.8.498.95728724960383464194568391302477328450\', \'1.2.826.0.1.3680043.8.498.65909865035783831607773206653044421345\', \'1.2.826.0.1.3680043.8.498.95638601079527306640829424921455011698\', \'1.2.826.0.1.3680043.8.498.35853404993418331599021266761280794201\', \'1.2.826.0.1.3680043.8.498.80627605463642274852766102874822833365\', \'1.2.826.0.1.3680043.8.498.12718966639501121738042125605376714231\', \'1.2.826.0.1.3680043.8.498.38617606118499305781729905815407382451\', \'1.2.826.0.1.3680043.8.498.93230313557251171119475173996942942855\', \'1.2.826.0.1.3680043.8.498.19198152070740205434848422424805073748\', \'1.2.826.0.1.3680043.8.498.40308838286515017547707130085914948230\', \'1.2.826.0.1.3680043.8.498.35027750475162395113025853006474122989\', \'1.2.826.0.1.3680043.8.498.70931968429039283348328653687636803315\', \'1.2.826.0.1.3680043.8.498.74749071555443582954158991248361308204\', \'1.2.826.0.1.3680043.8.498.11458488293461896930750018852127694875\', \'1.2.826.0.1.3680043.8.498.96448229695053562621553132798380456420\', \'1.2.826.0.1.3680043.8.498.64286017646659573966584053755363026715\', \'1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652\', \'1.2.826.0.1.3680043.8.498.99458163491875066451936719024359566928\', \'1.2.826.0.1.3680043.8.498.46776367027887983105817631582518625639\', \'1.2.826.0.1.3680043.8.498.34795373994453585401035013232069587413\', \'1.2.826.0.1.3680043.8.498.53579416035647672067965892020505016454\', \'1.2.826.0.1.3680043.8.498.12005327370242618323737652892230673473\', \'1.2.826.0.1.3680043.8.498.15219824433491621460467738389345697247\', \'1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334\', \'1.2.826.0.1.3680043.8.498.77513348456432770689938407624768928320\', \'1.2.826.0.1.3680043.8.498.28134711422142706763867845558971485241\', \'1.2.826.0.1.3680043.8.498.76065434429437758192282562351089010088\', \'1.2.826.0.1.3680043.8.498.13143446880514535170343243531129864116\', \'1.2.826.0.1.3680043.8.498.37591590536870291237852764995372956522\', \'1.2.826.0.1.3680043.8.498.11191015483427903713800997920940844550\', \'1.2.826.0.1.3680043.8.498.71985417286226511458068110832784023627\', \'1.2.826.0.1.3680043.8.498.98315778542448798493600485116505234132\', \'1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581\', \'1.2.826.0.1.3680043.8.498.96158169859491339496094295508194547199\', \'1.2.826.0.1.3680043.8.498.77299727102400166279263320763707637485\', \'1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091\', \'1.2.826.0.1.3680043.8.498.98107112397588450345153336193341117589\', \'1.2.826.0.1.3680043.8.498.10162168547770229150261479353227398478\', \'1.2.826.0.1.3680043.8.498.22398041676780235097751294563297953841\', \'1.2.826.0.1.3680043.8.498.32299120582647333604770626110392750631\', \'1.2.826.0.1.3680043.8.498.81974732979506931694079719601760393277\', \'1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317\', \'1.2.826.0.1.3680043.8.498.72977302282583186913749000016284027331\', \'1.2.826.0.1.3680043.8.498.90037507233074511524602897439257102032\', \'1.2.826.0.1.3680043.8.498.11786296135006283515679011075884245663\', \'1.2.826.0.1.3680043.8.498.49855677166670499939366963036804180912\', \'1.2.826.0.1.3680043.8.498.94945670991495145462605178471390637325\', \'1.2.826.0.1.3680043.8.498.80643480554795312280333659144941545925\', \'1.2.826.0.1.3680043.8.498.16843381581429596234606436682284624187\', \'1.2.826.0.1.3680043.8.498.21012635480218712816069264326295410954\', \'1.2.826.0.1.3680043.8.498.24313647347273496316728514161733307557\', \'1.2.826.0.1.3680043.8.498.63673733836218886397484265807806548473\', \'1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952\', \'1.2.826.0.1.3680043.8.498.77628479664122209492622263818114972176\', \'1.2.826.0.1.3680043.8.498.11465586454035959390661499120751232686\', \'1.2.826.0.1.3680043.8.498.90003472003278237920648198243464374280\', \'1.2.826.0.1.3680043.8.498.85943737587102032370951456635167782343\', \'1.2.826.0.1.3680043.8.498.13199635020145400856987271814458645896\', \'1.2.826.0.1.3680043.8.498.42018683052544325209168649309206539721\', \'1.2.826.0.1.3680043.8.498.41662013720606493874198348776219875496\', \'1.2.826.0.1.3680043.8.498.58298767414933858487578597918936525600\', \'1.2.826.0.1.3680043.8.498.12308357850075929758995438211317707857\', \'1.2.826.0.1.3680043.8.498.24291465113360632954006568021942978765\', \'1.2.826.0.1.3680043.8.498.10240237086492917349933491458808329326\', \'1.2.826.0.1.3680043.8.498.85719235937472088594405415004352845221\', \'1.2.826.0.1.3680043.8.498.12568337973304376222940287737691898395\', \'1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383\', \'1.2.826.0.1.3680043.8.498.53587920256280027683523152568832253423\', \'1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974\', \'1.2.826.0.1.3680043.8.498.10930374192038896613513882597332665403\', \'1.2.826.0.1.3680043.8.498.45368532788186860828867663606651818835\', \'1.2.826.0.1.3680043.8.498.35210451107308788915675910467662426761\', \'1.2.826.0.1.3680043.8.498.78119223228067919465880137763593966456\', \'1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197\', \'1.2.826.0.1.3680043.8.498.16101435445990180482665354366724730305\', \'1.2.826.0.1.3680043.8.498.42967970121147528831983176261453235744\', \'1.2.826.0.1.3680043.8.498.80247974926646352443043954025450139721\', \'1.2.826.0.1.3680043.8.498.80312559977515578735238310472949287111\', \'1.2.826.0.1.3680043.8.498.26875920364303755911342338961252130512\', \'1.2.826.0.1.3680043.8.498.79560170727391465844958390474997798803\', \'1.2.826.0.1.3680043.8.498.16189833935983604051983090678102023832\', \'1.2.826.0.1.3680043.8.498.40612173652156509478338094582396889655\', \'1.2.826.0.1.3680043.8.498.80284846390295834578273590480193235798\', \'1.2.826.0.1.3680043.8.498.29418130858182890216867847991592582370\', \'1.2.826.0.1.3680043.8.498.98022738778461179495257509215244053198\', \'1.2.826.0.1.3680043.8.498.11043926695471016727738735328317975282\', \'1.2.826.0.1.3680043.8.498.20232953944759707220316171133595554284\', \'1.2.826.0.1.3680043.8.498.51054592064718900209123281525185013961\', \'1.2.826.0.1.3680043.8.498.90483774039833567637730989859822082272\', \'1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724\', \'1.2.826.0.1.3680043.8.498.41382780862853765983216810855531499640\', \'1.2.826.0.1.3680043.8.498.94658317716039594639343437780115140531\', \'1.2.826.0.1.3680043.8.498.50680622091377404463269575858175673158\', \'1.2.826.0.1.3680043.8.498.10614508096225944703643482200835890514\', \'1.2.826.0.1.3680043.8.498.23409201792139701921731647470152180932\', \'1.2.826.0.1.3680043.8.498.89935384074170571164295050203650829722\', \'1.2.826.0.1.3680043.8.498.54761306437713414745748829128854749469\', \'1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414\', \'1.2.826.0.1.3680043.8.498.80290749690597272632725356393056746299\', \'1.2.826.0.1.3680043.8.498.77283634197289382453511269950488893300\', \'1.2.826.0.1.3680043.8.498.13284123714012951608998434802430211634\', \'1.2.826.0.1.3680043.8.498.92839397144213404004277739877191283401\', \'1.2.826.0.1.3680043.8.498.58604600600114899516493668210902251434\', \'1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172\', \'1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729\', \'1.2.826.0.1.3680043.8.498.43442933855807685981061557274166869461\', \'1.2.826.0.1.3680043.8.498.42476443240785038044326483794941265936\', \'1.2.826.0.1.3680043.8.498.12373923338033469765189220784089350380\', \'1.2.826.0.1.3680043.8.498.12553010776519233077752150342664160826\', \'1.2.826.0.1.3680043.8.498.50144715912379960437506930683683218333\', \'1.2.826.0.1.3680043.8.498.72313325634455670216423864504103231505\', \'1.2.826.0.1.3680043.8.498.47061252152535181371772020997295000029\', \'1.2.826.0.1.3680043.8.498.13126380916005729420163981055214890048\', \'1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558\', \'1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746\', \'1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166\', \'1.2.826.0.1.3680043.8.498.41564229176277158115386256329157870722\', \'1.2.826.0.1.3680043.8.498.47707670074001059720671838493302874710\', \'1.2.826.0.1.3680043.8.498.12946789387115203718210904958275579544\', \'1.2.826.0.1.3680043.8.498.74222294235046321147381553078000820604\', \'1.2.826.0.1.3680043.8.498.12307356432736970350794033171174403632\', \'1.2.826.0.1.3680043.8.498.71701484984624141610445774249629295610\', \'1.2.826.0.1.3680043.8.498.81832287065207835637503039077685315236\', \'1.2.826.0.1.3680043.8.498.42023788139360998656558751921918860808\', \'1.2.826.0.1.3680043.8.498.40967435550546093348221485942169879116\', \'1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434\', \'1.2.826.0.1.3680043.8.498.84477198486144701928775090913243250655\', \'1.2.826.0.1.3680043.8.498.82498616123349854719801627024073383874\', \'1.2.826.0.1.3680043.8.498.14400292895187181989838122627394761532\', \'1.2.826.0.1.3680043.8.498.87223930502803944549504594723314220105\', \'1.2.826.0.1.3680043.8.498.98541661035340329000785741171799559104\', \'1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982\', \'1.2.826.0.1.3680043.8.498.76914908131203373740345988718674850939\', \'1.2.826.0.1.3680043.8.498.45380312679924593883892686293576260616\', \'1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801\', \'1.2.826.0.1.3680043.8.498.16469256144078887607437578587242258846\', \'1.2.826.0.1.3680043.8.498.48367523758673249284754003765836877239\', \'1.2.826.0.1.3680043.8.498.10770877369291591553624215762368014357\', \'1.2.826.0.1.3680043.8.498.35677515667357567818845364400903668190\', \'1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755\', \'1.2.826.0.1.3680043.8.498.86435890312227285216810145658548164773\', \'1.2.826.0.1.3680043.8.498.11830915612184936576804778995164042200\', \'1.2.826.0.1.3680043.8.498.37447511763482014027579634343525200849\', \'1.2.826.0.1.3680043.8.498.12778920060586364664953350981285170949\', \'1.2.826.0.1.3680043.8.498.12333996498811456012744466468462396081\', \'1.2.826.0.1.3680043.8.498.18092766733837308394607532608319746043\', \'1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780\', \'1.2.826.0.1.3680043.8.498.11432815311029789072253610823554308183\', \'1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913\', \'1.2.826.0.1.3680043.8.498.85333821516157983299252782678788406019\', \'1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385\', \'1.2.826.0.1.3680043.8.498.68578659816221770558506780445608768600\', \'1.2.826.0.1.3680043.8.498.20411068889491114730824265411248922690\', \'1.2.826.0.1.3680043.8.498.25953117039631879376428439063599611036\', \'1.2.826.0.1.3680043.8.498.66973089869591928349547584554030589647\', \'1.2.826.0.1.3680043.8.498.99257151613846084492342083136797222759\', \'1.2.826.0.1.3680043.8.498.91860013313252950044036077977399321593\', \'1.2.826.0.1.3680043.8.498.11119324966854127911591363740049886324\', \'1.2.826.0.1.3680043.8.498.10519566075312736546346469231439794627\', \'1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531\', \'1.2.826.0.1.3680043.8.498.12818010040786942613541254450332754046\', \'1.2.826.0.1.3680043.8.498.12195007606427082866449558290311905939\', \'1.2.826.0.1.3680043.8.498.33657688895130083119886800524324572941\', \'1.2.826.0.1.3680043.8.498.38010459551726691601986492657296948013\', \'1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773\', \'1.2.826.0.1.3680043.8.498.67404282038417089622852480023919627784\', \'1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889\', \'1.2.826.0.1.3680043.8.498.32241219263926672894433308808653719495\', \'1.2.826.0.1.3680043.8.498.85454884397223412543050876117416146750\', \'1.2.826.0.1.3680043.8.498.65763261177572072944828171262268039852\', \'1.2.826.0.1.3680043.8.498.24369332907731805266450307536793220148\', \'1.2.826.0.1.3680043.8.498.70674155597869824035214737999044110827\', \'1.2.826.0.1.3680043.8.498.96564709711765982786409987515735443801\', \'1.2.826.0.1.3680043.8.498.24126422042626612740219325272793737656\', \'1.2.826.0.1.3680043.8.498.18350185433331570523543207086298834067\', \'1.2.826.0.1.3680043.8.498.16718788321951483456195507236392554704\', \'1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716\', \'1.2.826.0.1.3680043.8.498.46893956584698923808118623450380169212\', \'1.2.826.0.1.3680043.8.498.47949251060660360953300340380324578835\', \'1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295\', \'1.2.826.0.1.3680043.8.498.10960770430559425421609885784366918575\', \'1.2.826.0.1.3680043.8.498.61435096069921474761812731150911059491\', \'1.2.826.0.1.3680043.8.498.19871496221665809577186118817075295760\', \'1.2.826.0.1.3680043.8.498.54711709751232783991855540573189433446\', \'1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369\', \'1.2.826.0.1.3680043.8.498.12092744899746753313905432642371563786\', \'1.2.826.0.1.3680043.8.498.72252601583562621620422058467969183753\', \'1.2.826.0.1.3680043.8.498.95694598580744163428934772566469311180\', \'1.2.826.0.1.3680043.8.498.41240503248178304281405515398211762506\', \'1.2.826.0.1.3680043.8.498.25103708168569906591706406794385334861\', \'1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634\', \'1.2.826.0.1.3680043.8.498.55015061060346597564205577461227720715\', \'1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431\', \'1.2.826.0.1.3680043.8.498.17985756708438762670829959560049370730\', \'1.2.826.0.1.3680043.8.498.67001329390153622359718581388215312710\', \'1.2.826.0.1.3680043.8.498.10854223557359336882116168394037581766\', \'1.2.826.0.1.3680043.8.498.69729903409900053570304330127687235813\', \'1.2.826.0.1.3680043.8.498.11902464730418099436837163594625541861\', \'1.2.826.0.1.3680043.8.498.87679606849449562627719676100773778015\', \'1.2.826.0.1.3680043.8.498.18389676218363070462526838642303976332\', \'1.2.826.0.1.3680043.8.498.94252884278314596961426629318911909075\', \'1.2.826.0.1.3680043.8.498.28333127304382930639614800697959296306\', \'1.2.826.0.1.3680043.8.498.28697216332632397717077153882513537456\', \'1.2.826.0.1.3680043.8.498.78012314847102407375138809615977711653\', \'1.2.826.0.1.3680043.8.498.31827987476038668074615045841102711882\', \'1.2.826.0.1.3680043.8.498.30490857770169699336468319880130884989\', \'1.2.826.0.1.3680043.8.498.95570207430778332247200880037915693375\', \'1.2.826.0.1.3680043.8.498.20801986011906346655458394198951999413\', \'1.2.826.0.1.3680043.8.498.12497144849433082154223897356471317072\', \'1.2.826.0.1.3680043.8.498.11571057273590444914079281238002921868\', \'1.2.826.0.1.3680043.8.498.17123244662958741977525806700986256210\', \'1.2.826.0.1.3680043.8.498.92602515666185999130675702469977883616\', \'1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859\', \'1.2.826.0.1.3680043.8.498.94393348753936295687030819683853459606\', \'1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641\', \'1.2.826.0.1.3680043.8.498.12296624595750602443033689366743784387\', \'1.2.826.0.1.3680043.8.498.11529405479134457001825038259936796388\', \'1.2.826.0.1.3680043.8.498.87338030934656808032044071206920470285\', \'1.2.826.0.1.3680043.8.498.73257122462858353041066181004123253930\', \'1.2.826.0.1.3680043.8.498.61456481310419978302697858056095086811\', \'1.2.826.0.1.3680043.8.498.55713753582074871470943388423906540792\']\nassert set(ids)<=set(test.StudyInstanceUID)\nassert ids and len(ids)==len(set(ids))\nse=pd.read_csv(comp/\'train_series.csv\',dtype={\'StudyInstanceUID\':str,\'SeriesInstanceUID\':str})\nse=se[se.StudyInstanceUID.isin(ids)]\nn_files={sr:min(len(os.listdir(comp/\'train_series\'/st/sr)),MAX_SLICES) for st,sr in zip(se.StudyInstanceUID,se.SeriesInstanceUID)}\ntable=build_study_table(se,n_files)\nds=infer.DicomStudyDataset(ids,table,str(comp/\'train_series\'),12)\nmodels=infer.load_models([str(p) for p in ckpts])\nassert len(models)==3\nconfig=[]\nfor p in ckpts:\n    ck=torch.load(p,map_location=\'cpu\',weights_only=True)\n    assert ck[\'args\'][\'backbone\'].split(\'.\')[0]==\'convnext_tiny\' and int(ck[\'args\'][\'res\'])==256, \'Unexpected architecture/input\'\n    config.append(ck[\'args\'])\n    del ck\n# Full batch and all six slots: memory/latency check only, discarded after use.\nx,mask,pos=ds[0]\nbx=x.unsqueeze(0).repeat(4,1,1,1,1,1).cuda()\nbm=torch.ones((4,6),dtype=torch.bool,device=\'cuda\')\nbp=pos.unsqueeze(0).repeat(4,1,1).cuda()\nstress_start=time.monotonic()\nwith torch.inference_mode(), torch.autocast(\'cuda\',dtype=torch.float16):\n    for model,res in models:\n        probe=model(bx,bm,bp,res=res)\n        assert probe.shape==(4,12) and torch.isfinite(probe).all()\ntorch.cuda.synchronize()\nstress_seconds=time.monotonic()-stress_start\nstress_peak=torch.cuda.max_memory_allocated(0)\ndel bx,bm,bp,probe\ntorch.cuda.empty_cache()\n# Keep loading in-process: no CUDA-after-fork or shared-memory prefetch queue.\n# Batch size, windows, pixels and predictions are unchanged.\ndl=torch.utils.data.DataLoader(ds,batch_size=4,shuffle=False,num_workers=0,pin_memory=True)\nprint(\'CNX3 inference start\',len(ids),\'studies; loader_workers=0\',flush=True)\nimport resource\nout=[];valid=[];inference_start=time.monotonic()\nload_seconds=0.;gpu_seconds=0.;previous_end=inference_start\nprogress_path=Path(\'/kaggle/working/scale_progress.json\')\ndef save_progress(batch_index):\n    payload=dict(status=\'RUNNING\',requested=len(ids),processed=len(valid),batch_index=batch_index,\n        elapsed_seconds=time.monotonic()-started,load_seconds=load_seconds,gpu_seconds=gpu_seconds,\n        peak_rss_bytes=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss*1024,\n        peak_gpu_bytes=torch.cuda.max_memory_allocated(0),no_accuracy_evaluation=True)\n    tmp=progress_path.with_suffix(\'.tmp\');tmp.write_text(json.dumps(payload,indent=2));tmp.replace(progress_path)\nsave_progress(-1)\nwith torch.inference_mode():\n    for batch_index,(x,mask,pos) in enumerate(dl):\n        load_seconds+=time.monotonic()-previous_end\n        if time.monotonic()-started>3*3600:\n            save_progress(batch_index)\n            raise TimeoutError(\'Public scale diagnostic exceeded three-hour budget\')\n        gpu_start=time.monotonic()\n        assert mask.any(1).all(), \'Empty study\' \n        valid.extend(mask.sum(1).tolist())\n        x,mask,pos=x.cuda(non_blocking=True),mask.cuda(),pos.cuda()\n        with torch.autocast(\'cuda\',dtype=torch.float16):\n            prob=torch.stack([m(x,mask,pos,res=res).float().sigmoid() for m,res in models]).mean(0)\n        assert torch.isfinite(prob).all()\n        out.append(prob.cpu().numpy())\n        torch.cuda.synchronize()\n        gpu_seconds+=time.monotonic()-gpu_start\n        save_progress(batch_index)\n        if batch_index%25==0:\n            print(\'CNX3 progress\',len(valid),\'/\',len(ids),\'seconds\',time.monotonic()-inference_start,flush=True)\n        previous_end=time.monotonic()\nvalues=np.concatenate(out)\nassert values.shape==(len(ids),12) and np.isfinite(values).all() and (values>=0).all() and (values<=1).all()\nframe=pd.DataFrame(values,columns=LABELS)\nframe.insert(0,\'StudyInstanceUID\',ids)\nwork=Path(\'/kaggle/working\')\npath=work/\'scale_predictions.csv\'\nframe.to_csv(path,index=False)\nreceipt=dict(status=\'COMPLETE\',study_count=len(ids),folds=[0,1,2],model_count=3,\n    precision=\'fp16\',batch_studies=4,windows_per_slot=12,slots_present=valid,loader_workers=0,\n    checkpoint_hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in ckpts},\n    checkpoint_args=config,stress_pass=True,stress_seconds=stress_seconds,stress_peak_gpu_bytes=stress_peak,\n    peak_gpu_bytes=torch.cuda.max_memory_allocated(0),elapsed_seconds=time.monotonic()-started,\n    inference_seconds=time.monotonic()-inference_start,source_hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in src.glob(\'*.py\')},\n    raw_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),fallback=0,partial_predictions=0)\nreceipt.update(scope=\'public train execution only\',no_accuracy_evaluation=True,load_seconds=load_seconds,gpu_seconds=gpu_seconds)\n(work/\'scale_receipt.json\').write_text(json.dumps(receipt,indent=2,default=str))\nprint(\'CNX3 COMPLETE\',receipt,flush=True)\n'}.items(): (src/name).write_text(text,encoding='utf-8')
with open('/kaggle/working/scale_worker.log','w') as log:
    try:
        result=subprocess.run([sys.executable,'-u',str(src/'worker.py')],stdout=log,stderr=subprocess.STDOUT,timeout=3*3600+120)
        status='COMPLETE' if result.returncode==0 else 'ERROR'
        outcome=dict(status=status,returncode=result.returncode)
    except subprocess.TimeoutExpired:
        outcome=dict(status='BUDGET_LIMIT',seconds=3*3600+120)
Path('/kaggle/working/scale_outcome.json').write_text(json.dumps(outcome,indent=2))
print(outcome)
if outcome['status']!='COMPLETE':
    print(Path('/kaggle/working/scale_worker.log').read_text()[-8000:])
    raise RuntimeError('Public scale diagnostic did not complete; inspect preserved log and progress')
